# NLP-04 — 10_explainability

Publication copy of the research notebook.

Saved outputs and unnecessary metadata have been removed. Private annotations, raw datasets and model checkpoints are retained separately.

Some cells require external research artifacts. The public notebook is not a standalone reproduction package.

In [ ]:
# Cell 1: Explainability setup and artifact verification

from google.colab import drive
from pathlib import Path

import hashlib
import json
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

drive.mount('/content/drive')

PROJECT_ROOT = Path(
    '/content/drive/MyDrive/NLP04_Toxic_Comment_Classification'
)

assert PROJECT_ROOT.is_dir()

CONFIG_DIR = PROJECT_ROOT / '00_config'
SRC_DIR = PROJECT_ROOT / '03_src'
DATA_DIR = PROJECT_ROOT / '01_data'
PROCESSED_DIR = DATA_DIR / 'processed'
MODELS_DIR = PROJECT_ROOT / '04_models'

NOTEBOOK_DIR = (
    PROJECT_ROOT / '02_notebooks/10_explainability'
)

RESULTS_DIR = NOTEBOOK_DIR / 'results'

TABLES_DIR = RESULTS_DIR / 'tables'
FIGURES_DIR = RESULTS_DIR / 'figures'
LOGS_DIR = RESULTS_DIR / 'logs'
RUN_CONFIGS_DIR = RESULTS_DIR / 'configs'
PRIVATE_DIR = RESULTS_DIR / 'private_explanations'
HTML_DIR = PRIVATE_DIR / 'html'
ATTRIBUTIONS_DIR = PRIVATE_DIR / 'attributions'

for path in [
    TABLES_DIR,
    FIGURES_DIR,
    LOGS_DIR,
    RUN_CONFIGS_DIR,
    PRIVATE_DIR,
    HTML_DIR,
    ATTRIBUTIONS_DIR
]:
    path.mkdir(parents=True, exist_ok=True)

SEED = 42
rng = np.random.default_rng(SEED)

# File utility functions

def read_json(path):
    return json.loads(Path(path).read_text())


def calculate_sha256(path):

    digest = hashlib.sha256()

    with open(path, 'rb') as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024), b''
        ):
            digest.update(chunk)

    return digest.hexdigest()


def save_json_atomic(path, content):

    path = Path(path)
    temp = path.with_suffix(path.suffix + '.tmp')

    temp.write_text(
        json.dumps(content, indent=4, allow_nan=False)
    )

    assert read_json(temp) == content

    temp.replace(path)


def save_csv_atomic(path, frame):

    path = Path(path)
    temp = path.with_suffix(path.suffix + '.tmp')

    frame.to_csv(temp, index=False)

    temp.replace(path)


def verify_file(record):

    path = (PROJECT_ROOT / record['path']).resolve()

    assert path.is_relative_to(PROJECT_ROOT.resolve())
    assert path.is_file(), path
    assert path.stat().st_size == record['size_bytes']
    assert calculate_sha256(path) == record['sha256'], path

    return path


# Restore frozen model protocol

REGISTRY_PATH = CONFIG_DIR / 'experiment_registry.csv'
DECISION_LOG_PATH = CONFIG_DIR / 'project_decisions.csv'

FROZEN_MANIFEST_PATH = (
    PROJECT_ROOT /
    '02_notebooks/06_model_optimization/results/configs/'
    'frozen_manifest.json'
)

EXPECTED_FROZEN_HASH = (
    '229adb5d6166a626f6b7f7fddecf38b4'
    'de58d21e56ec4367465ad29174ef2fd1'
)

assert calculate_sha256(
    FROZEN_MANIFEST_PATH
) == EXPECTED_FROZEN_HASH

frozen_manifest = read_json(FROZEN_MANIFEST_PATH)

assert frozen_manifest['thresholds_frozen']
assert frozen_manifest['model_parameters_frozen']
assert not frozen_manifest['official_test_used_for_selection']

assert calculate_sha256(
    REGISTRY_PATH
) == frozen_manifest['registry']['sha256']

assert calculate_sha256(
    DECISION_LOG_PATH
) == frozen_manifest['decision_log_sha256']

LABELS = frozen_manifest['labels']

assert LABELS == [
    'toxic',
    'severe_toxic',
    'obscene',
    'threat',
    'insult',
    'identity_hate'
]

FROZEN_MODELS = {
    entry['run_id']: entry
    for entry in frozen_manifest['models']
}

assert len(FROZEN_MODELS) == 19

PRIMARY_RUN_ID = frozen_manifest[
    'primary_selected_run_id'
]

assert PRIMARY_RUN_ID == (
    'S02_ROBERTA_FOCAL_FULL_S42'
)

primary_model_entry = FROZEN_MODELS[PRIMARY_RUN_ID]

assert primary_model_entry['kind'] == 'transformer'

assert primary_model_entry['score_type'] == (
    'sigmoid_probability'
)

assert primary_model_entry['comparison_operator'] == '>='

assert primary_model_entry['maximum_length'] == 256

assert primary_model_entry['inference_precision'] == (
    'fp16_cuda'
)

thresholds = np.array([
    primary_model_entry['optimized_thresholds'][label]
    for label in LABELS
])

MODEL_PATH = PROJECT_ROOT / primary_model_entry['model_path']

assert MODEL_PATH.is_dir()

print('Frozen model protocol: Verified')

# Verify previous notebook completion records

EVALUATION_RESULTS_DIR = (
    PROJECT_ROOT /
    '02_notebooks/07_final_evaluation/results'
)

ERROR_RESULTS_DIR = (
    PROJECT_ROOT /
    '02_notebooks/08_error_analysis/results'
)

FAIRNESS_RESULTS_DIR = (
    PROJECT_ROOT /
    '02_notebooks/09_fairness_audit/results'
)

EVALUATION_COMPLETION_PATH = (
    EVALUATION_RESULTS_DIR /
    'logs/evaluation_completion_summary.json'
)

ERROR_COMPLETION_PATH = (
    ERROR_RESULTS_DIR /
    'logs/error_analysis_completion_summary.json'
)

FAIRNESS_COMPLETION_PATH = (
    FAIRNESS_RESULTS_DIR /
    'logs/fairness_audit_completion_summary.json'
)

evaluation_completion = read_json(
    EVALUATION_COMPLETION_PATH
)

error_completion = read_json(
    ERROR_COMPLETION_PATH
)

fairness_completion = read_json(
    FAIRNESS_COMPLETION_PATH
)

assert evaluation_completion['status'] == 'Completed'

assert evaluation_completion[
    'evaluated_configurations'
] == 19

assert error_completion['ai_assisted_review_completed']
assert error_completion['review_cases'] == 180

assert fairness_completion['status'] == 'Completed'
assert fairness_completion['synthetic_comments'] == 744
assert fairness_completion['verified_tables'] == 15
assert fairness_completion['verified_figures'] == 6

for record in [
    evaluation_completion,
    error_completion,
    fairness_completion
]:

    assert record['frozen_manifest_sha256'] == (
        EXPECTED_FROZEN_HASH
    )

    assert not record['thresholds_changed']
    assert not record['model_training_performed']
    assert not record['test_driven_selection_performed']

# Verify all inventoried fairness artifacts

fairness_inventory = fairness_completion[
    'artifact_inventory'
]

assert len(fairness_inventory) == 29

for record in fairness_inventory:
    verify_file(record)

print('Previous notebook completion records: Verified')
print('Fairness inventory: 29 artifacts verified')

# Restore official held-out test dataset

TEST_PATH = PROJECT_ROOT / evaluation_completion[
    'test_dataset_path'
]

EXPECTED_TEST_HASH = (
    'cddc70d876b48b9fe3d5db8109a7404'
    '30a7f4d9f50f9fcd77f4c7d566035f328'
)

assert calculate_sha256(
    TEST_PATH
) == EXPECTED_TEST_HASH

test_df = pd.read_parquet(
    TEST_PATH,
    columns=['id', 'comment_text', *LABELS]
)

assert len(test_df) == 63978
assert test_df['id'].is_unique
assert test_df['comment_text'].notna().all()

test_ids = test_df['id'].astype(str).to_numpy()
y_test = test_df[LABELS].to_numpy(dtype=np.int8)

assert np.isin(y_test, [0, 1]).all()

normalized_text = (
    test_df['comment_text']
    .str.lower()
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

test_group_ids, unique_groups = pd.factorize(
    normalized_text,
    sort=False
)

assert len(unique_groups) == 63939

del normalized_text, unique_groups
gc.collect()

# Restore existing frozen test predictions

inventory = {
    record['path']: record
    for record in evaluation_completion['artifact_inventory']
}

assert len(inventory) == 79


def verify_evaluation_artifact(path):

    key = str(
        Path(path).relative_to(PROJECT_ROOT)
    )

    return verify_file(inventory[key])


prediction_index_path = (
    PROJECT_ROOT /
    evaluation_completion['prediction_index_path']
)

verify_evaluation_artifact(prediction_index_path)

assert calculate_sha256(
    prediction_index_path
) == evaluation_completion['prediction_index_sha256']

prediction_index = pd.read_csv(
    prediction_index_path
).set_index('Run ID')

prediction_record = prediction_index.loc[
    PRIMARY_RUN_ID
]

primary_prediction_path = PROJECT_ROOT / (
    prediction_record['Prediction Path']
)

verify_evaluation_artifact(primary_prediction_path)

assert calculate_sha256(
    primary_prediction_path
) == prediction_record['SHA256']

primary_predictions = pd.read_parquet(
    primary_prediction_path
)

assert len(primary_predictions) == len(test_df)
assert primary_predictions['id'].is_unique

assert primary_predictions['run_id'].eq(
    PRIMARY_RUN_ID
).all()

assert primary_predictions['split'].eq(
    'test_official'
).all()

np.testing.assert_array_equal(
    primary_predictions['id'].astype(str).to_numpy(),
    test_ids
)

np.testing.assert_array_equal(
    primary_predictions[
        [f'true_{l}' for l in LABELS]
    ].to_numpy(),
    y_test
)

primary_scores = primary_predictions[
    [f'score_{l}' for l in LABELS]
].to_numpy(dtype=np.float64)

primary_pred = primary_predictions[
    [f'pred_optimized_{l}' for l in LABELS]
].to_numpy(dtype=np.int8)

assert np.isfinite(primary_scores).all()

assert (
    (primary_scores >= 0) &
    (primary_scores <= 1)
).all()

np.testing.assert_array_equal(
    primary_pred,
    (primary_scores >= thresholds).astype(np.int8)
)

print('Official test predictions: Verified')

# Restore frozen fairness identity lexicon

lexicon_path = (
    FAIRNESS_RESULTS_DIR /
    'configs/identity_lexicon.json'
)

LEXICON_HASH = (
    'df2cee3475216e5371391b0c464cdb150'
    'eb45bc8da6aac84da985c17703eb0a8'
)

assert calculate_sha256(lexicon_path) == LEXICON_HASH

assert fairness_completion['lexicon_sha256'] == (
    LEXICON_HASH
)

lexicon_config = read_json(lexicon_path)
identity_lexicon = lexicon_config['categories']

assert sum(
    len(terms)
    for terms in identity_lexicon.values()
) == 30

print('Frozen identity lexicon: Verified')

# Verify model files without loading the model

verify_file(primary_model_entry['config'])
verify_file(primary_model_entry['threshold_file'])

verify_file(
    primary_model_entry['checkpoint_manifest']
)

for record in primary_model_entry['model_files']:
    verify_file(record)

print('Frozen model and tokenizer files: Verified')

# Establish explanation methodology

explainability_setup = {

    'notebook': '10_explainability',
    'seed': SEED,
    'primary_run_id': PRIMARY_RUN_ID,
    'labels': LABELS,

    'method': 'LIME',
    'fallback_method': 'SHAP',

    'selection_partition': 'Official labelled test',

    'planned_cases': 20,

    'selection_quotas': {
        'true_positive': 6,
        'false_positive': 4,
        'false_negative': 4,
        'clean_identity_mention': 4,
        'challenging_multilabel': 2
    },

    'unique_comments_required': True,
    'six_label_coverage_required': True,

    'lime': {
        'initial_perturbations': 1000,
        'top_features': 10,
        'random_seed': SEED,
        'target_wrapper': '[1-p, p]',
        'one_target_label_per_case': True,
        'smoke_test_required': True,
        'record_local_surrogate_fidelity': True,
        'record_model_visible_features': True
    },

    'faithfulness': {
        'top_k': 5,
        'matched_random_draws': 20,
        'comprehensiveness': True,
        'sufficiency': True,
        'preserve_signed_score_changes': True
    },

    'maximum_token_length': 256,
    'inference_precision': 'fp16_cuda',
    'threshold_source': 'Frozen validation-tuned thresholds',

    'frozen_manifest_sha256': EXPECTED_FROZEN_HASH,
    'test_dataset_sha256': EXPECTED_TEST_HASH,
    'test_prediction_sha256': prediction_record['SHA256'],
    'lexicon_sha256': LEXICON_HASH,

    'evaluation_completion_sha256': calculate_sha256(
        EVALUATION_COMPLETION_PATH
    ),

    'error_completion_sha256': calculate_sha256(
        ERROR_COMPLETION_PATH
    ),

    'fairness_completion_sha256': calculate_sha256(
        FAIRNESS_COMPLETION_PATH
    ),

    'case_selection_completed': False,
    'model_training_performed': False,
    'thresholds_changed': False,
    'test_driven_selection_performed': False,
    'additional_inference_performed': False,

    'private_raw_comment_artifacts': True,

    'limitations': [
        'Selected cases are diagnostic, not representative.',
        'LIME approximates local model behaviour.',
        'Perturbed text may be linguistically unnatural.',
        'Feature weights do not establish causal mechanisms.',
        'Model scores are not assumed to be calibrated probabilities.'
    ]
}

setup_path = (
    RUN_CONFIGS_DIR / 'explainability_setup.json'
)

if setup_path.exists():

    assert read_json(setup_path) == explainability_setup

else:

    save_json_atomic(
        setup_path,
        explainability_setup
    )

# Summarize available prediction outcomes

outcome_summary = pd.DataFrame({
    'Label': LABELS,
    'Frozen Threshold': thresholds,

    'True Positives': (
        (primary_pred == 1) & (y_test == 1)
    ).sum(axis=0),

    'False Positives': (
        (primary_pred == 1) & (y_test == 0)
    ).sum(axis=0),

    'False Negatives': (
        (primary_pred == 0) & (y_test == 1)
    ).sum(axis=0),

    'True Negatives': (
        (primary_pred == 0) & (y_test == 0)
    ).sum(axis=0)
})

save_csv_atomic(
    TABLES_DIR / 'explanation_candidate_summary.csv',
    outcome_summary
)

print('\nAvailable explanation candidates')

display(outcome_summary.round(6))

# Final protocol checks

assert calculate_sha256(
    FROZEN_MANIFEST_PATH
) == EXPECTED_FROZEN_HASH

assert calculate_sha256(
    REGISTRY_PATH
) == frozen_manifest['registry']['sha256']

assert calculate_sha256(
    DECISION_LOG_PATH
) == frozen_manifest['decision_log_sha256']

print('\nExplainability setup summary')
print('Selected model:', PRIMARY_RUN_ID)
print('Official held-out comments:', len(test_df))
print('Normalized comment groups:', len(np.unique(test_group_ids)))
print('Target labels:', len(LABELS))
print('Planned explanation cases:', 20)
print('Initial LIME perturbations per case:', 1000)
print('Random deletion draws per case:', 20)
print('Maximum token length:', 256)
print('Frozen model files: Verified')
print('Fairness artifacts verified:', len(fairness_inventory))
print('Model loading and inference: None')
print('Frozen protocol: Unchanged')
print('Cell 1 completed successfully.')

In [ ]:
# Cell 2: Select and freeze 20 explanation cases

import re

from transformers import AutoTokenizer

# Load tokenizer only; no model loading or inference
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    use_fast=True
)

assert tokenizer.is_fast
assert tokenizer.truncation_side == 'right'

MAX_LENGTH = primary_model_entry['maximum_length']

selection_rng = np.random.default_rng(SEED)

# Restore identity-term matching patterns
identity_patterns = {}

for category, terms in identity_lexicon.items():

    identity_patterns[category] = {}

    for term, variants in terms.items():

        alternatives = [
            re.escape(v).replace(r'\ ', r'\s+')
            for v in sorted(
                variants,
                key=len,
                reverse=True
            )
        ]

        pattern = re.compile(
            r'(?<!\w)(?:' +
            '|'.join(alternatives) +
            r')(?!\w)',
            re.IGNORECASE
        )

        identity_patterns[category][term] = pattern

# Construct identity category masks

identity_category_masks = {}

for category, patterns in identity_patterns.items():

    masks = [
        test_df['comment_text'].str.contains(
            pattern,
            regex=True,
            na=False
        ).to_numpy(dtype=bool)

        for pattern in patterns.values()
    ]

    identity_category_masks[category] = (
        np.logical_or.reduce(masks)
    )

any_identity_mask = np.logical_or.reduce(
    list(identity_category_masks.values())
)

assert any_identity_mask.sum() == 4763

# Construct prediction outcome masks

tp_mask = (y_test == 1) & (primary_pred == 1)
fp_mask = (y_test == 0) & (primary_pred == 1)
fn_mask = (y_test == 1) & (primary_pred == 0)

all_label_clean = (y_test == 0).all(axis=1)
all_label_unflagged = (primary_pred == 0).all(axis=1)

multilabel_mask = y_test.sum(axis=1) >= 2

partially_correct_mask = tp_mask.any(axis=1)

selected_rows = []
selected_groups = set()

# Helper: obtain identity categories and terms

def find_identity_mentions(text):

    categories = []
    terms_found = []

    for category, patterns in identity_patterns.items():

        matches = [
            term
            for term, pattern in patterns.items()
            if pattern.search(text)
        ]

        if matches:

            categories.append(category)
            terms_found.extend(matches)

    return categories, terms_found


# Reproducible sampling within predefined strata

def select_case(
    stratum,
    mask,
    target_label=None,
    target_error=None,
    selection_detail='',
    require_truncated=False
):

    candidates = np.flatnonzero(mask)

    candidates = selection_rng.permutation(
        candidates
    )

    selected_index = None
    selected_token_count = None
    selected_word_count = None

    length_fallback_used = False

    length_modes = (
        ['truncated', 'untruncated']
        if require_truncated
        else ['untruncated']
    )

    for length_mode in length_modes:

        for index in candidates:

            index = int(index)

            group_id = int(test_group_ids[index])

            if group_id in selected_groups:
                continue

            text = test_df.iloc[index]['comment_text']

            # Ensure adequate distinct LIME word features
            words = re.findall(r'\w+', text)

            distinct_words = len(set(words))

            if distinct_words < 8:
                continue

            token_ids = tokenizer(
                text,
                truncation=False,
                add_special_tokens=True,
                return_attention_mask=False,
                verbose=False
            )['input_ids']

            token_count = len(token_ids)

            if length_mode == 'truncated':

                # Include one manageable truncated example
                if not MAX_LENGTH < token_count <= 512:
                    continue

            else:

                if token_count > MAX_LENGTH:
                    continue

            selected_index = index
            selected_token_count = token_count
            selected_word_count = distinct_words

            length_fallback_used = (
                require_truncated and
                length_mode == 'untruncated'
            )

            break

        if selected_index is not None:
            break

    if selected_index is None:

        raise RuntimeError(
            f'No suitable candidate found: {stratum}, '
            f'{selection_detail}'
        )

    index = selected_index

    # Determine the target for challenging multi-label cases

    if target_label is None:

        if target_error == 'FN':

            available = np.flatnonzero(
                fn_mask[index]
            )

        elif target_error == 'FP':

            available = np.flatnonzero(
                fp_mask[index]
            )

        else:

            raise ValueError(
                'A target label or error type is required.'
            )

        assert len(available) > 0

        target_label = LABELS[int(available[0])]

    label_index = LABELS.index(target_label)

    actual = int(y_test[index, label_index])

    predicted = int(
        primary_pred[index, label_index]
    )

    score = float(
        primary_scores[index, label_index]
    )

    threshold = float(thresholds[label_index])

    assert predicted == int(score >= threshold)

    outcome = {
        (1, 1): 'TP',
        (0, 1): 'FP',
        (1, 0): 'FN',
        (0, 0): 'TN'
    }[(actual, predicted)]

    text = test_df.iloc[index]['comment_text']

    categories, terms_found = find_identity_mentions(
        text
    )

    actual_labels = [
        label
        for j, label in enumerate(LABELS)
        if y_test[index, j] == 1
    ]

    predicted_labels = [
        label
        for j, label in enumerate(LABELS)
        if primary_pred[index, j] == 1
    ]

    case_id = f'E{len(selected_rows) + 1:02d}'

    selected_rows.append({
        'Case ID': case_id,
        'Stratum': stratum,
        'Selection Detail': selection_detail,

        'Comment ID': str(test_ids[index]),
        'Source Row': index,

        'Normalized Group ID': int(
            test_group_ids[index]
        ),

        'Target Label': target_label,
        'Target Outcome': outcome,

        'Actual Target': actual,
        'Predicted Target': predicted,
        'Target Score': score,
        'Frozen Threshold': threshold,
        'Score Minus Threshold': score - threshold,

        'Actual Positive Labels': (
            ' | '.join(actual_labels)
            if actual_labels else 'None'
        ),

        'Predicted Positive Labels': (
            ' | '.join(predicted_labels)
            if predicted_labels else 'None'
        ),

        'Actual Positive Count': len(actual_labels),
        'Predicted Positive Count': len(predicted_labels),

        'Identity Mention': bool(terms_found),

        'Identity Categories': (
            ' | '.join(categories)
            if categories else 'None'
        ),

        'Identity Terms': (
            ' | '.join(terms_found)
            if terms_found else 'None'
        ),

        'Distinct Word Features': selected_word_count,
        'Original Token Count': selected_token_count,

        'Visible Token Count': min(
            selected_token_count,
            MAX_LENGTH
        ),

        'Truncated': (
            selected_token_count > MAX_LENGTH
        ),

        'Length Fallback Used': length_fallback_used,

        'Candidate Pool Size': len(candidates),

        'Comment Text': text
    })

    selected_groups.add(
        int(test_group_ids[index])
    )

    print(
        case_id,
        '|', stratum,
        '|', target_label,
        '|', outcome,
        '| Tokens:', selected_token_count
    )


# Part A: Six true positives, one per label

for j, label in enumerate(LABELS):

    select_case(
        stratum='True positive',
        mask=tp_mask[:, j],
        target_label=label,
        selection_detail=f'True positive for {label}'
    )


# Part B: Four false positives

FP_TARGETS = [
    'toxic',
    'severe_toxic',
    'threat',
    'identity_hate'
]

for label in FP_TARGETS:

    j = LABELS.index(label)

    select_case(
        stratum='False positive',
        mask=fp_mask[:, j],
        target_label=label,
        selection_detail=f'False positive for {label}'
    )


# Part C: Four false negatives

FN_TARGETS = [
    'toxic',
    'threat',
    'insult',
    'identity_hate'
]

for label in FN_TARGETS:

    j = LABELS.index(label)

    select_case(
        stratum='False negative',
        mask=fn_mask[:, j],
        target_label=label,
        selection_detail=f'False negative for {label}'
    )


# Part D: Four clean identity-containing comments

toxic_index = LABELS.index('toxic')

# Two incorrectly flagged identity-containing comments
# and two correctly unflagged comments

clean_identity_design = [

    ('Sexual orientation', 'FP'),

    ('Sexual orientation', 'TN'),

    ('Religion', 'FP'),

    ('Gender', 'TN')
]

for category, expected_outcome in clean_identity_design:

    mask = (
        all_label_clean &
        identity_category_masks[category]
    )

    if expected_outcome == 'FP':

        mask = (
            mask &
            fp_mask[:, toxic_index]
        )

    else:

        mask = (
            mask &
            all_label_unflagged
        )

    select_case(
        stratum='Clean identity mention',
        mask=mask,
        target_label='toxic',
        selection_detail=(
            f'{category}: {expected_outcome}'
        )
    )


# Part E: Two challenging multi-label comments

# One missed label in a partially correct prediction

select_case(
    stratum='Challenging multilabel',

    mask=(
        multilabel_mask &
        partially_correct_mask &
        fn_mask.any(axis=1)
    ),

    target_error='FN',

    selection_detail=(
        'Partially correct multilabel prediction '
        'with a missed label'
    )
)

# One extra incorrect label, preferably in truncated text

select_case(
    stratum='Challenging multilabel',

    mask=(
        multilabel_mask &
        partially_correct_mask &
        fp_mask.any(axis=1)
    ),

    target_error='FP',

    require_truncated=True,

    selection_detail=(
        'Partially correct multilabel prediction '
        'with an extra incorrect label'
    )
)


# Verify the final selected explanation cases

selected_cases = pd.DataFrame(selected_rows)

assert len(selected_cases) == 20

assert selected_cases['Case ID'].is_unique
assert selected_cases['Comment ID'].is_unique

assert selected_cases[
    'Normalized Group ID'
].is_unique

assert selected_cases['Comment Text'].is_unique

assert set(
    selected_cases['Target Label']
) == set(LABELS)

expected_quotas = {
    'True positive': 6,
    'False positive': 4,
    'False negative': 4,
    'Clean identity mention': 4,
    'Challenging multilabel': 2
}

assert selected_cases[
    'Stratum'
].value_counts().to_dict() == expected_quotas

assert selected_cases[
    'Distinct Word Features'
].ge(8).all()

# Verify selected cases against frozen test predictions

for row in selected_cases.to_dict('records'):

    index = row['Source Row']
    j = LABELS.index(row['Target Label'])

    assert row['Comment ID'] == str(test_ids[index])

    assert row['Comment Text'] == (
        test_df.iloc[index]['comment_text']
    )

    assert row['Actual Target'] == y_test[index, j]

    assert row['Predicted Target'] == (
        primary_pred[index, j]
    )

    assert row['Target Score'] == (
        primary_scores[index, j]
    )

    assert row['Frozen Threshold'] == thresholds[j]

    if row['Stratum'] == 'True positive':
        assert row['Target Outcome'] == 'TP'

    elif row['Stratum'] == 'False positive':
        assert row['Target Outcome'] == 'FP'

    elif row['Stratum'] == 'False negative':
        assert row['Target Outcome'] == 'FN'

    elif row['Stratum'] == 'Clean identity mention':

        assert y_test[index].sum() == 0
        assert row['Identity Mention']

    elif row['Stratum'] == 'Challenging multilabel':

        assert y_test[index].sum() >= 2
        assert tp_mask[index].any()

        assert (
            fp_mask[index].any() or
            fn_mask[index].any()
        )

print('\nSelected cases and frozen predictions: Verified')


# Save selected comments privately

selected_cases_path = (
    PRIVATE_DIR / 'T01_selected_comments.csv'
)

# Publication-safe metadata excludes raw comments
# and original dataset identifiers

public_case_metadata = selected_cases.drop(
    columns=[
        'Comment Text',
        'Comment ID',
        'Source Row',
        'Normalized Group ID'
    ]
)

metadata_path = (
    TABLES_DIR / 'T01_selected_case_metadata.csv'
)

selection_summary = (
    selected_cases.groupby(
        ['Stratum', 'Target Outcome']
    )
    .size()
    .reset_index(name='Cases')
)

selection_summary_path = (
    TABLES_DIR / 'explanation_selection_summary.csv'
)


# Prevent silent replacement of selected cases

def save_or_verify_selection(path, frame):

    expected_bytes = frame.to_csv(
        index=False
    ).encode('utf-8')

    expected_hash = hashlib.sha256(
        expected_bytes
    ).hexdigest()

    if path.exists():

        assert calculate_sha256(path) == expected_hash, (
            f'Existing frozen selection differs: {path}'
        )

    else:

        save_csv_atomic(path, frame)

    assert calculate_sha256(path) == expected_hash


save_or_verify_selection(
    selected_cases_path,
    selected_cases
)

save_or_verify_selection(
    metadata_path,
    public_case_metadata
)

save_or_verify_selection(
    selection_summary_path,
    selection_summary
)

SELECTION_HASH = calculate_sha256(
    selected_cases_path
)


# Record case selection methodology before LIME

selection_config = {

    'status': 'Completed',

    'seed': SEED,
    'primary_run_id': PRIMARY_RUN_ID,

    'selection_partition': 'Official labelled test',

    'selected_cases': 20,

    'selection_quotas': expected_quotas,

    'sampling_method': (
        'Seeded random candidate ordering '
        'within predefined diagnostic strata'
    ),

    'unique_normalized_comment_groups': True,
    'minimum_distinct_word_features': 8,

    'true_positive_targets': LABELS,
    'false_positive_targets': FP_TARGETS,
    'false_negative_targets': FN_TARGETS,

    'clean_identity_design': [
        {
            'category': category,
            'target_outcome': outcome
        }
        for category, outcome in clean_identity_design
    ],

    'multilabel_selection': {
        'minimum_actual_positive_labels': 2,
        'requires_at_least_one_true_positive': True,
        'target_errors': ['FN', 'FP'],
        'target_selection_rule': (
            'First eligible error in fixed label order'
        )
    },

    'length_protocol': {
        'ordinary_cases_maximum_tokens': MAX_LENGTH,
        'prefer_one_truncated_multilabel_case': True,
        'truncated_case_maximum_tokens': 512,

        'truncated_cases_selected': int(
            selected_cases['Truncated'].sum()
        ),

        'length_fallback_cases': int(
            selected_cases['Length Fallback Used'].sum()
        )
    },

    'selected_case_ids': (
        selected_cases['Case ID'].tolist()
    ),

    'selected_cases_path': str(
        selected_cases_path.relative_to(PROJECT_ROOT)
    ),

    'selected_cases_sha256': SELECTION_HASH,

    'public_metadata_path': str(
        metadata_path.relative_to(PROJECT_ROOT)
    ),

    'public_metadata_sha256': calculate_sha256(
        metadata_path
    ),

    'selection_summary_sha256': calculate_sha256(
        selection_summary_path
    ),

    'frozen_manifest_sha256': EXPECTED_FROZEN_HASH,
    'test_dataset_sha256': EXPECTED_TEST_HASH,

    'test_prediction_sha256': calculate_sha256(
        primary_prediction_path
    ),

    'lexicon_sha256': LEXICON_HASH,

    'lime_explanations_examined_before_selection': False,
    'model_training_performed': False,
    'thresholds_changed': False,
    'additional_inference_performed': False,

    'limitations': [
        'Diagnostic cases are not a representative test sample.',
        'Selection depends on predefined prediction outcomes.',
        'Minimum word count excludes very short comments.',
        'Length restrictions favour interpretable input windows.',
        'Identity mentions are lexical proxies.',
        'Dataset annotations may contain disagreement or ambiguity.'
    ]
}

selection_config_path = (
    RUN_CONFIGS_DIR /
    'explanation_case_selection.json'
)

if selection_config_path.exists():

    assert read_json(
        selection_config_path
    ) == selection_config

else:

    save_json_atomic(
        selection_config_path,
        selection_config
    )


# Display selected case metadata without raw toxic text

print('\nSelected explanation cases')

display(
    selected_cases[[
        'Case ID',
        'Stratum',
        'Target Label',
        'Target Outcome',
        'Target Score',
        'Frozen Threshold',
        'Actual Positive Count',
        'Predicted Positive Count',
        'Original Token Count',
        'Truncated'
    ]].round(6)
)

print('\nCase distribution')

display(selection_summary)

# Confirm frozen protocol remains unchanged

assert calculate_sha256(
    FROZEN_MANIFEST_PATH
) == EXPECTED_FROZEN_HASH

assert calculate_sha256(
    REGISTRY_PATH
) == frozen_manifest['registry']['sha256']

assert calculate_sha256(
    DECISION_LOG_PATH
) == frozen_manifest['decision_log_sha256']

assert calculate_sha256(
    primary_prediction_path
) == prediction_record['SHA256']

print('\nCell 2 summary')

print('Selected cases:', len(selected_cases))

print(
    'Unique comments:',
    selected_cases['Comment ID'].nunique()
)

print(
    'Unique normalized groups:',
    selected_cases['Normalized Group ID'].nunique()
)

print(
    'Target label coverage:',
    selected_cases['Target Label'].nunique()
)

print(
    'Truncated cases:',
    selected_cases['Truncated'].sum()
)

print(
    'Length fallback cases:',
    selected_cases['Length Fallback Used'].sum()
)

print('Selection SHA-256:', SELECTION_HASH)
print('Private selected comments: Saved')
print('Publication-safe metadata: Saved')
print('Selection configuration: Saved')
print('LIME inference: Not started')
print('Frozen protocol: Unchanged')
print('Cell 2 completed successfully.')

del tokenizer
gc.collect()

In [ ]:
# Cell 3: Frozen model loading and LIME smoke test

import os
import sys
import time
import gc
import subprocess
import importlib.metadata

import torch
import transformers

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

assert torch.cuda.is_available(), 'Enable T4 GPU first.'

DEVICE = torch.device('cuda')

os.environ.setdefault(
    'CUBLAS_WORKSPACE_CONFIG', ':4096:8'
)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

print('GPU:', torch.cuda.get_device_name(0))
print('Torch:', torch.__version__)
print('Transformers:', transformers.__version__)

# Install LIME without changing existing dependencies

LIME_VERSION = '0.2.0.1'

try:
    installed_version = importlib.metadata.version('lime')
except importlib.metadata.PackageNotFoundError:
    installed_version = None

if installed_version != LIME_VERSION:

    subprocess.check_call([
        sys.executable,
        '-m', 'pip', 'install',
        '--quiet',
        '--no-deps',
        f'lime=={LIME_VERSION}'
    ])

from lime.lime_text import (
    LimeTextExplainer,
    IndexedString
)

assert importlib.metadata.version('lime') == LIME_VERSION

print('LIME version:', LIME_VERSION)

# Restore frozen explanation cases

selection_config_path = (
    RUN_CONFIGS_DIR /
    'explanation_case_selection.json'
)

selection_config = read_json(selection_config_path)

selected_cases_path = (
    PROJECT_ROOT /
    selection_config['selected_cases_path']
)

SELECTION_HASH = (
    'b440129b98122fbcb89f1fc039a88fb38'
    '27affe3ea3ff6a7be64670721d45f92'
)

assert calculate_sha256(
    selected_cases_path
) == SELECTION_HASH

assert selection_config[
    'selected_cases_sha256'
] == SELECTION_HASH

assert selection_config[
    'frozen_manifest_sha256'
] == EXPECTED_FROZEN_HASH

selected_cases = pd.read_csv(
    selected_cases_path,
    keep_default_na=False,
    float_precision='round_trip'
)

assert len(selected_cases) == 20
assert selected_cases['Case ID'].is_unique
assert selected_cases['Comment ID'].is_unique

# Reverify all selected cases against original predictions

for row in selected_cases.to_dict('records'):

    i = int(row['Source Row'])
    j = LABELS.index(row['Target Label'])

    assert row['Comment ID'] == str(test_ids[i])

    assert row['Comment Text'] == (
        test_df.iloc[i]['comment_text']
    )

    assert row['Target Score'] == primary_scores[i, j]

    assert row['Frozen Threshold'] == thresholds[j]

    assert row['Actual Target'] == y_test[i, j]

    assert row['Predicted Target'] == primary_pred[i, j]

print('Frozen explanation cases: Verified')

# Load the original tokenizer and model

for record in primary_model_entry['model_files']:
    verify_file(record)

checkpoint_manifest = read_json(
    verify_file(primary_model_entry['checkpoint_manifest'])
)

assert checkpoint_manifest['model_state_sha256'] == (
    primary_model_entry['model_state_sha256']
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    use_fast=True
)

assert tokenizer.is_fast
assert tokenizer.padding_side == 'right'
assert tokenizer.truncation_side == 'right'

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    local_files_only=True
)

assert model.config.num_labels == len(LABELS)

# Verify loaded model state

def model_state_hash(model):

    digest = hashlib.sha256()

    for name, tensor in sorted(
        model.state_dict().items()
    ):

        tensor = tensor.detach().cpu().contiguous()

        digest.update(name.encode('utf-8'))
        digest.update(str(tensor.dtype).encode('utf-8'))
        digest.update(str(tuple(tensor.shape)).encode('utf-8'))
        digest.update(tensor.numpy().tobytes())

    return digest.hexdigest()


assert model_state_hash(model) == (
    primary_model_entry['model_state_sha256']
)

model = model.to(DEVICE).eval()

model.requires_grad_(False)

assert not model.training

MAX_LENGTH = primary_model_entry['maximum_length']
PAD_MULTIPLE = primary_model_entry['padding_multiple']

BATCH_SIZE = 32

assert MAX_LENGTH == 256
assert PAD_MULTIPLE == 8

print('Frozen RoBERTa model state: Verified')

# Original frozen inference procedure

@torch.inference_mode()
def predict_batch(texts):

    encoded = tokenizer(
        texts,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
        add_special_tokens=True,
        return_attention_mask=False,
        return_token_type_ids=False
    )

    features = [
        {
            'input_ids': ids,
            'attention_mask': [1] * len(ids)
        }
        for ids in encoded['input_ids']
    ]

    batch = tokenizer.pad(
        features,
        padding=True,
        pad_to_multiple_of=PAD_MULTIPLE,
        return_tensors='pt'
    )

    assert batch['input_ids'].shape[1] <= MAX_LENGTH

    with torch.autocast(
        device_type='cuda',
        dtype=torch.float16
    ):

        logits = model(
            input_ids=batch['input_ids'].to(DEVICE),
            attention_mask=batch['attention_mask'].to(DEVICE)
        ).logits.float()

    scores = torch.sigmoid(logits)

    return (
        logits.cpu().numpy().astype(np.float64),
        scores.cpu().numpy().astype(np.float64)
    )


def predict_multilabel(texts):

    texts = list(texts)

    if not texts:
        return np.empty((0, len(LABELS)))

    batches = []

    for start in range(0, len(texts), BATCH_SIZE):

        _, scores = predict_batch(
            texts[start:start + BATCH_SIZE]
        )

        batches.append(scores)

    result = np.concatenate(batches, axis=0)

    assert result.shape == (len(texts), len(LABELS))
    assert np.isfinite(result).all()
    assert ((result >= 0) & (result <= 1)).all()

    return result


# Replay original test batches containing selected cases

replay_starts = sorted({
    int(i) // BATCH_SIZE * BATCH_SIZE
    for i in selected_cases['Source Row']
})

replay_score_error = 0.0
replay_logit_error = 0.0
replay_disagreements = 0
replay_samples = 0

for start in replay_starts:

    end = min(start + BATCH_SIZE, len(test_df))

    observed_logits, observed_scores = predict_batch(
        test_df.iloc[start:end]['comment_text'].tolist()
    )

    expected_scores = primary_scores[start:end]

    expected_logits = primary_predictions.iloc[
        start:end
    ][[
        f'logit_{label}' for label in LABELS
    ]].to_numpy(dtype=np.float64)

    replay_score_error = max(
        replay_score_error,
        float(np.max(np.abs(
            observed_scores - expected_scores
        )))
    )

    replay_logit_error = max(
        replay_logit_error,
        float(np.max(np.abs(
            observed_logits - expected_logits
        )))
    )

    replay_disagreements += int(
        (
            (observed_scores >= thresholds) !=
            (expected_scores >= thresholds)
        ).sum()
    )

    replay_samples += end - start

# Retain previous cross-runtime tolerances

assert replay_score_error <= 0.002, replay_score_error
assert replay_logit_error <= 0.02, replay_logit_error

print('\nOriginal prediction replay: Passed')
print('Replay samples:', replay_samples)
print('Maximum score error:', replay_score_error)
print('Maximum logit error:', replay_logit_error)
print('Prediction disagreements:', replay_disagreements)

# Fix LIME settings before generating explanations

LIME_SAMPLES = 1000
LIME_FEATURES = 10
LIME_KERNEL_WIDTH = 25
LIME_SPLIT_EXPRESSION = r'\W+'

lime_settings = {
    'method': 'LIME',
    'lime_version': LIME_VERSION,
    'num_samples': LIME_SAMPLES,
    'maximum_explanation_features': LIME_FEATURES,
    'kernel_width': LIME_KERNEL_WIDTH,
    'split_expression': LIME_SPLIT_EXPRESSION,
    'bow': True,
    'feature_selection': 'auto',
    'distance_metric': 'cosine',
    'target_class_index': 1,
    'binary_wrapper': '[1-p, p]',
    'seed_rule': '42 + case number - 1',
    'maximum_token_length': MAX_LENGTH,
    'padding_multiple': PAD_MULTIPLE,
    'batch_size': BATCH_SIZE,
    'inference_precision': 'fp16_cuda',
    'selection_sha256': SELECTION_HASH,
    'frozen_manifest_sha256': EXPECTED_FROZEN_HASH
}

lime_settings_path = (
    RUN_CONFIGS_DIR / 'lime_settings.json'
)

if lime_settings_path.exists():

    assert read_json(lime_settings_path) == lime_settings

else:

    save_json_atomic(
        lime_settings_path,
        lime_settings
    )

# Label-specific binary wrapper for multi-label LIME

def make_lime_predictor(target_label):

    j = LABELS.index(target_label)

    def classifier_fn(texts):

        scores = predict_multilabel(texts)

        p = scores[:, j]

        result = np.column_stack([
            1.0 - p,
            p
        ])

        assert result.shape == (len(texts), 2)

        np.testing.assert_allclose(
            result.sum(axis=1),
            1.0,
            atol=1e-12
        )

        return result

    return classifier_fn


# Check feature visibility in the original input window

def get_feature_visibility(
    text, indexed_string, feature_id, visible_offsets
):

    word = indexed_string.word(feature_id)

    positions = indexed_string.string_position(
        feature_id
    )

    visible_occurrences = 0
    fully_visible_occurrences = 0

    for position in positions:

        start = int(position)
        end = start + len(word)

        covered = np.zeros(len(word), dtype=bool)

        for left, right in visible_offsets:

            overlap_start = max(start, left)
            overlap_end = min(end, right)

            if overlap_end > overlap_start:

                covered[
                    overlap_start - start:
                    overlap_end - start
                ] = True

        visible_occurrences += int(covered.any())

        fully_visible_occurrences += int(
            covered.all()
        )

    total_occurrences = len(positions)

    if visible_occurrences == 0:
        visibility = 'Outside original input window'

    elif fully_visible_occurrences == total_occurrences:
        visibility = 'Fully visible'

    else:
        visibility = 'Partially visible'

    return {
        'Total Occurrences': total_occurrences,
        'Visible Occurrences': visible_occurrences,
        'Fully Visible Occurrences': fully_visible_occurrences,
        'Original Window Visibility': visibility
    }


# Reusable explanation function for Cells 3 and 4

def explain_case(row):

    case_id = row['Case ID']
    text = row['Comment Text']
    target = row['Target Label']

    case_seed = SEED + int(case_id[1:]) - 1

    classifier_fn = make_lime_predictor(target)

    indexed = IndexedString(
        text,
        split_expression=LIME_SPLIT_EXPRESSION,
        bow=True
    )

    assert indexed.num_words() >= 8

    explainer = LimeTextExplainer(
        class_names=[
            f'Not {target}',
            target
        ],
        kernel_width=LIME_KERNEL_WIDTH,
        feature_selection='auto',
        split_expression=LIME_SPLIT_EXPRESSION,
        bow=True,
        random_state=case_seed
    )

    torch.cuda.synchronize()

    started = time.perf_counter()

    explanation = explainer.explain_instance(
        text_instance=text,
        classifier_fn=classifier_fn,
        labels=(1,),
        num_features=min(
            LIME_FEATURES,
            indexed.num_words()
        ),
        num_samples=LIME_SAMPLES,
        distance_metric='cosine'
    )

    torch.cuda.synchronize()

    elapsed = time.perf_counter() - started

    original_score = float(
        explanation.predict_proba[1]
    )

    frozen_score = float(row['Target Score'])

    score_error = abs(
        original_score - frozen_score
    )

    assert score_error <= 0.002, (
        case_id, score_error
    )

    cutoff = float(row['Frozen Threshold'])

    assert int(original_score >= cutoff) == (
        int(row['Predicted Target'])
    )

    assert np.isfinite(explanation.score)

    local_prediction = float(
        np.asarray(explanation.local_pred).ravel()[0]
    )

    assert np.isfinite(local_prediction)

    # Original visible token offsets

    visible_encoding = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        add_special_tokens=True,
        return_offsets_mapping=True
    )

    visible_offsets = [
        (int(left), int(right))
        for left, right in visible_encoding['offset_mapping']
        if right > left
    ]

    feature_rows = []

    lime_indexed = (
        explanation.domain_mapper.indexed_string
    )

    feature_weights = explanation.as_map()[1]

    assert len(feature_weights) > 0

    for rank, (feature_id, weight) in enumerate(
        feature_weights,
        start=1
    ):

        feature_id = int(feature_id)
        weight = float(weight)

        assert np.isfinite(weight)

        assert lime_indexed.word(feature_id) == (
            indexed.word(feature_id)
        )

        visibility = get_feature_visibility(
            text,
            lime_indexed,
            feature_id,
            visible_offsets
        )

        feature_rows.append({
            'Case ID': case_id,
            'Target Label': target,
            'Target Outcome': row['Target Outcome'],
            'Feature Rank': rank,
            'Feature ID': feature_id,
            'Feature': lime_indexed.word(feature_id),
            'LIME Weight': weight,
            'Absolute Weight': abs(weight),
            'Direction': (
                'Increases target score'
                if weight > 0
                else 'Decreases target score'
                if weight < 0
                else 'Zero weight'
            ),
            **visibility
        })

    feature_frame = pd.DataFrame(feature_rows)

    summary = {
        'case_id': case_id,
        'target_label': target,
        'target_outcome': row['Target Outcome'],
        'frozen_score': frozen_score,
        'lime_original_score': original_score,
        'original_score_error': score_error,
        'frozen_threshold': cutoff,
        'surrogate_r2': float(explanation.score),
        'surrogate_prediction': local_prediction,
        'surrogate_intercept': float(
            explanation.intercept[1]
        ),
        'distinct_lime_features': indexed.num_words(),
        'explained_features': len(feature_frame),
        'positive_features': int(
            (feature_frame['LIME Weight'] > 0).sum()
        ),
        'negative_features': int(
            (feature_frame['LIME Weight'] < 0).sum()
        ),
        'outside_window_features': int(
            (
                feature_frame['Original Window Visibility'] ==
                'Outside original input window'
            ).sum()
        ),
        'num_samples': LIME_SAMPLES,
        'case_seed': case_seed,
        'inference_seconds': elapsed,
        'selection_sha256': SELECTION_HASH,
        'lime_settings_sha256': calculate_sha256(
            lime_settings_path
        )
    }

    return explanation, feature_frame, summary


# Smoke test on the first frozen explanation case

smoke_case = selected_cases[
    selected_cases['Case ID'] == 'E01'
].iloc[0]

assert smoke_case['Target Label'] == 'toxic'

print('\nRunning LIME smoke test on E01...')

smoke_explanation, smoke_features, smoke_summary = (
    explain_case(smoke_case)
)

# Save the smoke explanation for reuse in Cell 4

smoke_feature_path = (
    ATTRIBUTIONS_DIR /
    'E01_feature_attributions.csv'
)

smoke_html_path = HTML_DIR / 'E01_explanation.html'

smoke_summary_path = (
    LOGS_DIR / 'lime_smoke_test_summary.json'
)

save_csv_atomic(
    smoke_feature_path,
    smoke_features
)

smoke_explanation.save_to_file(
    str(smoke_html_path),
    labels=[1]
)

assert smoke_html_path.stat().st_size > 0

smoke_summary.update({
    'status': 'Passed',
    'feature_path': str(
        smoke_feature_path.relative_to(PROJECT_ROOT)
    ),
    'feature_sha256': calculate_sha256(
        smoke_feature_path
    ),
    'html_path': str(
        smoke_html_path.relative_to(PROJECT_ROOT)
    ),
    'html_sha256': calculate_sha256(
        smoke_html_path
    ),
    'replay_samples': replay_samples,
    'replay_score_error': replay_score_error,
    'replay_logit_error': replay_logit_error,
    'replay_prediction_disagreements': replay_disagreements,
    'gpu': torch.cuda.get_device_name(0),
    'torch_version': str(torch.__version__),
    'transformers_version': transformers.__version__,
    'model_training_performed': False,
    'thresholds_changed': False
})

save_json_atomic(
    smoke_summary_path,
    smoke_summary
)

# Display non-textual smoke test results

print('\nLIME smoke test results')

display(pd.DataFrame([{
    'Case ID': smoke_summary['case_id'],
    'Target Label': smoke_summary['target_label'],
    'Frozen Score': smoke_summary['frozen_score'],
    'LIME Original Score': smoke_summary['lime_original_score'],
    'Score Error': smoke_summary['original_score_error'],
    'Surrogate R2': smoke_summary['surrogate_r2'],
    'Surrogate Prediction': smoke_summary['surrogate_prediction'],
    'Explained Features': smoke_summary['explained_features'],
    'Positive Features': smoke_summary['positive_features'],
    'Negative Features': smoke_summary['negative_features'],
    'Perturbation Samples': LIME_SAMPLES,
    'Elapsed Seconds': smoke_summary['inference_seconds']
}]).round(6))

# Verify frozen research protocol

assert calculate_sha256(
    FROZEN_MANIFEST_PATH
) == EXPECTED_FROZEN_HASH

assert calculate_sha256(
    REGISTRY_PATH
) == frozen_manifest['registry']['sha256']

assert calculate_sha256(
    DECISION_LOG_PATH
) == frozen_manifest['decision_log_sha256']

assert calculate_sha256(
    selected_cases_path
) == SELECTION_HASH

print('\nCell 3 summary')
print('Frozen model weights: Verified')
print('Original prediction replay: Passed')
print('LIME smoke test: Passed')
print('Explained cases: 1 / 20')
print('Perturbation samples:', LIME_SAMPLES)
print('Private feature attribution CSV: Saved')
print('Private HTML explanation: Saved')
print('Model training: None')
print('Frozen protocol: Unchanged')
print('Cell 3 completed successfully.')

In [ ]:
# Cell 4: Generate and save all 20 LIME explanations

from PIL import Image
from pathlib import Path
from datetime import datetime, timezone

assert torch.cuda.is_available()
assert not model.training
assert LIME_SAMPLES == 1000
assert len(selected_cases) == 20

# Verify the frozen selection and model protocol

assert calculate_sha256(
    selected_cases_path
) == SELECTION_HASH

assert calculate_sha256(
    FROZEN_MANIFEST_PATH
) == EXPECTED_FROZEN_HASH

LIME_SETTINGS_HASH = calculate_sha256(
    lime_settings_path
)

# Keep individual explanations private

CASE_LOGS_DIR = PRIVATE_DIR / 'case_logs'
PRIVATE_FIGURES_DIR = PRIVATE_DIR / 'figures'

for path in [
    CASE_LOGS_DIR,
    PRIVATE_FIGURES_DIR
]:
    path.mkdir(parents=True, exist_ok=True)

# Restore verified smoke test

smoke_summary = read_json(
    LOGS_DIR / 'lime_smoke_test_summary.json'
)

assert smoke_summary['status'] == 'Passed'
assert smoke_summary['case_id'] == 'E01'
assert smoke_summary['num_samples'] == LIME_SAMPLES

assert smoke_summary['selection_sha256'] == SELECTION_HASH

assert smoke_summary[
    'lime_settings_sha256'
] == LIME_SETTINGS_HASH

assert calculate_sha256(
    PROJECT_ROOT / smoke_summary['feature_path']
) == smoke_summary['feature_sha256']

assert calculate_sha256(
    PROJECT_ROOT / smoke_summary['html_path']
) == smoke_summary['html_sha256']

print('E01 smoke test artifacts: Verified')


# Verify individual case results

def verify_case_features(frame, summary, row):

    case_id = row['Case ID']

    assert summary['case_id'] == case_id

    assert summary['target_label'] == row['Target Label']

    assert summary['target_outcome'] == row['Target Outcome']

    assert summary['frozen_score'] == float(
        row['Target Score']
    )

    assert summary['selection_sha256'] == SELECTION_HASH

    assert summary[
        'lime_settings_sha256'
    ] == LIME_SETTINGS_HASH

    assert summary['num_samples'] == LIME_SAMPLES

    assert len(frame) == summary['explained_features']

    assert 1 <= len(frame) <= LIME_FEATURES

    assert frame['Case ID'].eq(case_id).all()

    assert frame['Target Label'].eq(
        row['Target Label']
    ).all()

    assert frame['Feature ID'].is_unique

    assert np.isfinite(
        frame['LIME Weight']
    ).all()

    assert np.isfinite(summary['surrogate_r2'])

    assert summary['original_score_error'] <= 0.002

    assert int(
        summary['lime_original_score'] >=
        summary['frozen_threshold']
    ) == int(row['Predicted Target'])

    np.testing.assert_allclose(
        frame['Absolute Weight'],
        frame['LIME Weight'].abs(),
        atol=1e-12
    )

    assert (
        frame['Fully Visible Occurrences'] <=
        frame['Visible Occurrences']
    ).all()

    assert (
        frame['Visible Occurrences'] <=
        frame['Total Occurrences']
    ).all()


# Generate individual explanation figures

def create_explanation_figure(frame, summary, path):

    plot_frame = frame.sort_values(
        'Absolute Weight',
        ascending=True
    )

    weights = plot_frame['LIME Weight'].to_numpy()

    words = plot_frame['Feature'].astype(str).tolist()

    colors = [
        '#C65D49' if value > 0 else '#326796'
        for value in weights
    ]

    fig, ax = plt.subplots(
        figsize=(9, 5.5)
    )

    ax.barh(
        np.arange(len(weights)),
        weights,
        color=colors,
        alpha=0.9
    )

    ax.set_yticks(np.arange(len(weights)))
    ax.set_yticklabels(words)

    ax.axvline(
        0,
        color='black',
        linewidth=0.8
    )

    ax.set_xlabel(
        'Signed LIME feature weight'
    )

    ax.set_title(
        f"{summary['case_id']} — "
        f"{summary['target_label']} "
        f"({summary['target_outcome']})"
    )

    fig.text(
        0.5,
        0.01,
        (
            f"Model score: {summary['lime_original_score']:.4f}"
            f" | Threshold: {summary['frozen_threshold']:.4f}"
            f" | Local R²: {summary['surrogate_r2']:.4f}"
        ),
        ha='center',
        fontsize=10
    )

    ax.grid(axis='x', alpha=0.2)
    ax.set_axisbelow(True)

    fig.tight_layout(
        rect=[0, 0.045, 1, 1]
    )

    temp_path = path.with_suffix('.tmp.png')

    fig.savefig(
        temp_path,
        dpi=300,
        bbox_inches='tight',
        facecolor='white'
    )

    plt.close(fig)

    with Image.open(temp_path) as image:
        assert image.format == 'PNG'
        image.verify()

    temp_path.replace(path)


# Generate or restore every selected explanation

all_feature_frames = []
all_case_summaries = []

reused_cases = 0
newly_explained_cases = 0

for _, row in selected_cases.iterrows():

    case_id = row['Case ID']

    print('\nProcessing:', case_id, flush=True)

    feature_path = (
        ATTRIBUTIONS_DIR /
        f'{case_id}_feature_attributions.csv'
    )

    html_path = (
        HTML_DIR /
        f'{case_id}_explanation.html'
    )

    figure_path = (
        PRIVATE_FIGURES_DIR /
        f'{case_id}_explanation.png'
    )

    case_log_path = (
        CASE_LOGS_DIR /
        f'{case_id}_summary.json'
    )

    # Resume completed cases without additional inference

    if case_log_path.exists():

        summary = read_json(case_log_path)

        assert summary['status'] == 'Completed'

        for key, path in [
            ('feature_sha256', feature_path),
            ('html_sha256', html_path),
            ('figure_sha256', figure_path)
        ]:

            assert calculate_sha256(path) == summary[key]

        feature_frame = pd.read_csv(
            feature_path,
            keep_default_na=False,
            dtype={'Feature': str},
            float_precision='round_trip'
        )

        reused_cases += 1

        print('Existing completed case: Verified')

    else:

        # Reuse successful smoke explanation

        if case_id == 'E01':

            summary = dict(smoke_summary)

            feature_frame = pd.read_csv(
                feature_path,
                keep_default_na=False,
                dtype={'Feature': str},
                float_precision='round_trip'
            )

            reused_cases += 1

            print('Reusing verified smoke explanation')

        else:

            # Generate LIME explanation

            explanation, feature_frame, summary = (
                explain_case(row)
            )

            verify_case_features(
                feature_frame,
                summary,
                row
            )

            save_csv_atomic(
                feature_path,
                feature_frame
            )

            temp_html = html_path.with_suffix(
                '.tmp.html'
            )

            explanation.save_to_file(
                str(temp_html),
                labels=[1]
            )

            assert temp_html.stat().st_size > 0

            temp_html.replace(html_path)

            newly_explained_cases += 1

            del explanation

        # Verify stored feature attributions

        verify_case_features(
            feature_frame,
            summary,
            row
        )

        # Save private individual figure

        create_explanation_figure(
            feature_frame,
            summary,
            figure_path
        )

        # Save case-level completion record

        summary.update({

            'status': 'Completed',

            'feature_path': str(
                feature_path.relative_to(PROJECT_ROOT)
            ),

            'feature_sha256': calculate_sha256(
                feature_path
            ),

            'html_path': str(
                html_path.relative_to(PROJECT_ROOT)
            ),

            'html_sha256': calculate_sha256(
                html_path
            ),

            'figure_path': str(
                figure_path.relative_to(PROJECT_ROOT)
            ),

            'figure_sha256': calculate_sha256(
                figure_path
            ),

            'model_training_performed': False,
            'thresholds_changed': False
        })

        save_json_atomic(
            case_log_path,
            summary
        )

    # Verify case before adding to combined results

    verify_case_features(
        feature_frame,
        summary,
        row
    )

    all_feature_frames.append(feature_frame)
    all_case_summaries.append(summary)

    print(
        case_id,
        '| Label:', summary['target_label'],
        '| R2:', round(summary['surrogate_r2'], 4),
        '| Features:', summary['explained_features'],
        '| Score error:', round(
            summary['original_score_error'], 6
        ),
        '| Seconds:', round(
            summary['inference_seconds'], 3
        ),
        flush=True
    )

    gc.collect()


# Consolidate all feature attributions

feature_attributions = pd.concat(
    all_feature_frames,
    ignore_index=True
)

assert feature_attributions[
    'Case ID'
].nunique() == 20

assert not feature_attributions.duplicated(
    ['Case ID', 'Feature ID']
).any()

assert len(all_case_summaries) == 20

assert len(feature_attributions) == sum(
    record['explained_features']
    for record in all_case_summaries
)

attribution_path = (
    PRIVATE_DIR /
    'T02_feature_attributions.csv'
)

save_csv_atomic(
    attribution_path,
    feature_attributions
)


# Build publication-safe explanation summary

summary_rows = []

for record in all_case_summaries:

    summary_rows.append({

        'Case ID': record['case_id'],
        'Target Label': record['target_label'],
        'Target Outcome': record['target_outcome'],

        'Frozen Score': record['frozen_score'],

        'LIME Original Score': (
            record['lime_original_score']
        ),

        'Original Score Error': (
            record['original_score_error']
        ),

        'Frozen Threshold': (
            record['frozen_threshold']
        ),

        'Local Surrogate R2': (
            record['surrogate_r2']
        ),

        'Surrogate Prediction': (
            record['surrogate_prediction']
        ),

        'Surrogate Absolute Error': abs(
            record['surrogate_prediction'] -
            record['lime_original_score']
        ),

        'Distinct LIME Features': (
            record['distinct_lime_features']
        ),

        'Explained Features': (
            record['explained_features']
        ),

        'Positive Features': (
            record['positive_features']
        ),

        'Negative Features': (
            record['negative_features']
        ),

        'Outside Window Features': (
            record['outside_window_features']
        ),

        'Perturbation Samples': (
            record['num_samples']
        ),

        'Inference Seconds': (
            record['inference_seconds']
        )
    })

lime_case_summary = pd.DataFrame(summary_rows)

assert len(lime_case_summary) == 20

assert lime_case_summary['Case ID'].is_unique

assert lime_case_summary[
    'Original Score Error'
].le(0.002).all()

summary_path = (
    TABLES_DIR /
    'lime_case_summary.csv'
)

save_csv_atomic(
    summary_path,
    lime_case_summary
)


# Summarize local surrogate fidelity

fidelity_summary = pd.DataFrame([{

    'Explained Cases': len(lime_case_summary),

    'Mean R2': lime_case_summary[
        'Local Surrogate R2'
    ].mean(),

    'Median R2': lime_case_summary[
        'Local Surrogate R2'
    ].median(),

    'Minimum R2': lime_case_summary[
        'Local Surrogate R2'
    ].min(),

    'Maximum R2': lime_case_summary[
        'Local Surrogate R2'
    ].max(),

    'Mean Surrogate Absolute Error': (
        lime_case_summary[
            'Surrogate Absolute Error'
        ].mean()
    ),

    'Maximum Original Score Error': (
        lime_case_summary[
            'Original Score Error'
        ].max()
    ),

    'Total Perturbation Samples': int(
        lime_case_summary[
            'Perturbation Samples'
        ].sum()
    ),

    'Total Feature Attributions': len(
        feature_attributions
    ),

    'Outside Window Features': int(
        lime_case_summary[
            'Outside Window Features'
        ].sum()
    ),

    'Total Explanation Seconds': (
        lime_case_summary[
            'Inference Seconds'
        ].sum()
    )
}])

fidelity_path = (
    TABLES_DIR /
    'lime_fidelity_summary.csv'
)

save_csv_atomic(
    fidelity_path,
    fidelity_summary
)


# Verify all explanation artifacts

explanation_inventory = []

for record in all_case_summaries:

    case_id = record['case_id']

    for path_key, hash_key in [
        ('feature_path', 'feature_sha256'),
        ('html_path', 'html_sha256'),
        ('figure_path', 'figure_sha256')
    ]:

        path = PROJECT_ROOT / record[path_key]

        assert path.is_file()
        assert path.stat().st_size > 0

        assert calculate_sha256(path) == record[hash_key]

        explanation_inventory.append({
            'path': str(path.relative_to(PROJECT_ROOT)),
            'size_bytes': int(path.stat().st_size),
            'sha256': record[hash_key]
        })

    case_log_path = (
        CASE_LOGS_DIR /
        f'{case_id}_summary.json'
    )

    explanation_inventory.append({
        'path': str(
            case_log_path.relative_to(PROJECT_ROOT)
        ),

        'size_bytes': int(
            case_log_path.stat().st_size
        ),

        'sha256': calculate_sha256(
            case_log_path
        )
    })

assert len(explanation_inventory) == 80

for path in [
    attribution_path,
    summary_path,
    fidelity_path
]:

    explanation_inventory.append({
        'path': str(path.relative_to(PROJECT_ROOT)),
        'size_bytes': int(path.stat().st_size),
        'sha256': calculate_sha256(path)
    })

assert len(explanation_inventory) == 83

for record in explanation_inventory:
    verify_file(record)


# Verify the model protocol remained unchanged

assert calculate_sha256(
    FROZEN_MANIFEST_PATH
) == EXPECTED_FROZEN_HASH

assert calculate_sha256(
    REGISTRY_PATH
) == frozen_manifest['registry']['sha256']

assert calculate_sha256(
    DECISION_LOG_PATH
) == frozen_manifest['decision_log_sha256']

assert calculate_sha256(
    selected_cases_path
) == SELECTION_HASH

assert calculate_sha256(
    lime_settings_path
) == LIME_SETTINGS_HASH

assert not model.training

assert model_state_hash(model) == (
    primary_model_entry['model_state_sha256']
)


# Record explanation generation completion

lime_completion = {

    'status': 'Completed',
    'method': 'LIME',
    'primary_run_id': PRIMARY_RUN_ID,

    'explained_cases': 20,

    'target_labels_covered': int(
        lime_case_summary['Target Label'].nunique()
    ),

    'perturbations_per_case': LIME_SAMPLES,

    'total_perturbation_samples': int(
        lime_case_summary[
            'Perturbation Samples'
        ].sum()
    ),

    'feature_attribution_records': len(
        feature_attributions
    ),

    'html_explanations': 20,
    'individual_explanation_figures': 20,
    'case_completion_records': 20,

    'mean_surrogate_r2': float(
        lime_case_summary[
            'Local Surrogate R2'
        ].mean()
    ),

    'minimum_surrogate_r2': float(
        lime_case_summary[
            'Local Surrogate R2'
        ].min()
    ),

    'maximum_original_score_error': float(
        lime_case_summary[
            'Original Score Error'
        ].max()
    ),

    'frozen_manifest_sha256': EXPECTED_FROZEN_HASH,

    'selection_sha256': SELECTION_HASH,

    'lime_settings_sha256': LIME_SETTINGS_HASH,

    'attribution_path': str(
        attribution_path.relative_to(PROJECT_ROOT)
    ),

    'attribution_sha256': calculate_sha256(
        attribution_path
    ),

    'summary_path': str(
        summary_path.relative_to(PROJECT_ROOT)
    ),

    'summary_sha256': calculate_sha256(
        summary_path
    ),

    'artifact_inventory': explanation_inventory,

    'gpu': torch.cuda.get_device_name(0),

    'model_training_performed': False,
    'thresholds_changed': False,
    'additional_inference_performed': True,

    'faithfulness_analysis_completed': False,
    'written_interpretations_completed': False,

    'private_individual_explanations': True,

    'completed_at': datetime.now(
        timezone.utc
    ).isoformat()
}

save_json_atomic(
    LOGS_DIR / 'lime_generation_completion.json',
    lime_completion
)


# Display final explanation results

print('\nLIME explanation results')

display(
    lime_case_summary[[
        'Case ID',
        'Target Label',
        'Target Outcome',
        'Frozen Score',
        'LIME Original Score',
        'Original Score Error',
        'Local Surrogate R2',
        'Surrogate Absolute Error',
        'Explained Features',
        'Outside Window Features',
        'Inference Seconds'
    ]].round(6)
)

print('\nOverall local surrogate fidelity')

display(fidelity_summary.round(6))

print('\nCell 4 summary')
print('Explained cases:', len(lime_case_summary))
print('Reused explanations:', reused_cases)
print('Newly generated explanations:', newly_explained_cases)
print('Feature attribution records:', len(feature_attributions))
print('Perturbation samples:', 20 * LIME_SAMPLES)
print('HTML explanations saved:', 20)
print('Individual PNG figures saved:', 20)
print('Case completion records:', 20)
print('Verified inventory artifacts:', len(explanation_inventory))
print('Model weights: Unchanged')
print('Model training: None')
print('Frozen protocol: Unchanged')
print('Cell 4 completed successfully.')

In [ ]:
# Cell 5: LIME faithfulness evaluation

from datetime import datetime, timezone

assert torch.cuda.is_available()
assert not model.training
assert len(selected_cases) == 20

TOP_K = 5
RANDOM_DRAWS = 20

# Restore verified LIME artifacts

lime_completion_path = (
    LOGS_DIR / 'lime_generation_completion.json'
)

lime_completion = read_json(lime_completion_path)

assert lime_completion['status'] == 'Completed'
assert lime_completion['explained_cases'] == 20

assert lime_completion['selection_sha256'] == SELECTION_HASH

assert lime_completion[
    'frozen_manifest_sha256'
] == EXPECTED_FROZEN_HASH

assert len(lime_completion['artifact_inventory']) == 83

for record in lime_completion['artifact_inventory']:
    verify_file(record)

assert calculate_sha256(
    selected_cases_path
) == SELECTION_HASH

assert calculate_sha256(
    lime_settings_path
) == lime_completion['lime_settings_sha256']

attribution_path = (
    PROJECT_ROOT / lime_completion['attribution_path']
)

feature_attributions = pd.read_csv(
    attribution_path,
    keep_default_na=False,
    dtype={'Feature': str},
    float_precision='round_trip'
)

assert len(feature_attributions) == 196

print('LIME explanation artifacts: Verified')

# Fix the faithfulness experiment configuration

faithfulness_config = {
    'method': 'LIME feature perturbation',
    'seed': SEED,
    'top_k': TOP_K,
    'random_draws_per_case': RANDOM_DRAWS,

    'feature_ranking': (
        'Descending absolute LIME weight; '
        'feature ID ascending for ties'
    ),

    'perturbation_operator': (
        'LIME IndexedString.inverse_removing'
    ),

    'bow': True,

    'random_sampling': (
        'Without replacement within each draw; '
        'independent draws may repeat subsets'
    ),

    'random_seed_rule': (
        'SEED + 1000 + case number'
    ),

    'signed_score_changes_preserved': True,

    'absolute_score_change_reported': True,

    'comprehensiveness': (
        'Original score minus top-deletion score'
    ),

    'sufficiency': (
        'Original score minus top-retention score'
    ),

    'selection_sha256': SELECTION_HASH,

    'attribution_sha256': calculate_sha256(
        attribution_path
    ),

    'frozen_manifest_sha256': EXPECTED_FROZEN_HASH,

    'lime_settings_sha256': calculate_sha256(
        lime_settings_path
    ),

    'inference_precision': 'fp16_cuda',

    'limitations': [
        'Top-ranked features may contain both positive '
        'and negative LIME weights.',
        'Removing words can create unnatural sentences.',
        'Repeated word occurrences share a BOW feature.',
        'Deletion can expose previously truncated text.',
        'Scores are not assumed to be calibrated.',
        'Diagnostic cases are not representative.'
    ]
}

faithfulness_config_path = (
    RUN_CONFIGS_DIR / 'faithfulness_config.json'
)

if faithfulness_config_path.exists():

    assert read_json(
        faithfulness_config_path
    ) == faithfulness_config

else:

    save_json_atomic(
        faithfulness_config_path,
        faithfulness_config
    )


# Part A: Evaluate each explanation

faithfulness_rows = []
perturbation_rows = []

torch.cuda.synchronize()
experiment_start = time.perf_counter()

for _, row in selected_cases.iterrows():

    case_id = row['Case ID']
    target = row['Target Label']
    j = LABELS.index(target)

    text = row['Comment Text']

    cutoff = float(row['Frozen Threshold'])
    frozen_score = float(row['Target Score'])

    case_features = feature_attributions[
        feature_attributions['Case ID'] == case_id
    ].copy()

    assert len(case_features) > 0
    assert case_features['Target Label'].eq(target).all()

    # Restore the exact LIME word-feature representation

    indexed = IndexedString(
        text,
        split_expression=LIME_SPLIT_EXPRESSION,
        bow=True
    )

    feature_count = indexed.num_words()

    assert feature_count >= 8

    for feature in case_features.to_dict('records'):

        feature_id = int(feature['Feature ID'])

        assert 0 <= feature_id < feature_count

        assert indexed.word(feature_id) == (
            feature['Feature']
        )

    # Select top features using absolute LIME weights

    ranked = case_features.sort_values(
        ['Absolute Weight', 'Feature ID'],
        ascending=[False, True]
    )

    k = min(
        TOP_K,
        len(ranked),
        feature_count
    )

    top_features = ranked.head(k)

    top_ids = top_features[
        'Feature ID'
    ].to_numpy(dtype=int)

    assert len(np.unique(top_ids)) == k

    all_ids = np.arange(feature_count)

    remaining_ids = np.setdiff1d(
        all_ids,
        top_ids
    )

    # Use the same removal operator as LIME

    top_deleted_text = indexed.inverse_removing(
        top_ids
    )

    top_retained_text = indexed.inverse_removing(
        remaining_ids
    )

    assert top_deleted_text != text
    assert top_retained_text != text

    # Construct matched random-deletion controls

    case_rng = np.random.default_rng(
        SEED + 1000 + int(case_id[1:])
    )

    random_sets = []
    random_texts = []

    for draw in range(RANDOM_DRAWS):

        random_ids = np.sort(
            case_rng.choice(
                all_ids,
                size=k,
                replace=False
            )
        )

        assert len(np.unique(random_ids)) == k

        random_sets.append(random_ids)

        random_texts.append(
            indexed.inverse_removing(random_ids)
        )

    # Evaluate original and perturbed texts together

    texts = [
        text,
        top_deleted_text,
        top_retained_text,
        *random_texts
    ]

    assert len(texts) == 23

    scores = predict_multilabel(texts)[:, j]

    assert len(scores) == 23
    assert np.isfinite(scores).all()
    assert ((scores >= 0) & (scores <= 1)).all()

    original_score = float(scores[0])
    top_deleted_score = float(scores[1])
    retained_score = float(scores[2])

    random_scores = scores[3:]

    assert len(random_scores) == RANDOM_DRAWS

    # Verify the original prediction

    replay_error = abs(
        original_score - frozen_score
    )

    assert replay_error <= 0.002, (
        case_id, replay_error
    )

    assert int(original_score >= cutoff) == (
        int(row['Predicted Target'])
    )

    # Calculate signed and absolute faithfulness measures

    comprehensiveness = (
        original_score - top_deleted_score
    )

    sufficiency = (
        original_score - retained_score
    )

    random_drops = (
        original_score - random_scores
    )

    mean_random_drop = float(
        random_drops.mean()
    )

    mean_random_absolute_change = float(
        np.abs(random_drops).mean()
    )

    signed_advantage = (
        comprehensiveness - mean_random_drop
    )

    absolute_advantage = (
        abs(comprehensiveness) -
        mean_random_absolute_change
    )

    original_pred = int(
        original_score >= cutoff
    )

    top_deleted_pred = int(
        top_deleted_score >= cutoff
    )

    retained_pred = int(
        retained_score >= cutoff
    )

    random_pred = (
        random_scores >= cutoff
    ).astype(int)

    positive_top_features = int(
        (top_features['LIME Weight'] > 0).sum()
    )

    negative_top_features = int(
        (top_features['LIME Weight'] < 0).sum()
    )

    unique_random_subsets = len({
        tuple(ids.tolist())
        for ids in random_sets
    })

    # Save aggregate case-level metrics without raw text

    faithfulness_rows.append({

        'Case ID': case_id,
        'Stratum': row['Stratum'],
        'Target Label': target,
        'Target Outcome': row['Target Outcome'],

        'Actual Target': int(row['Actual Target']),
        'Original Prediction': original_pred,

        'Frozen Score': frozen_score,
        'Original Score': original_score,
        'Original Score Error': replay_error,
        'Frozen Threshold': cutoff,

        'Distinct Features': feature_count,
        'Top K': k,

        'Positive Top Features': positive_top_features,
        'Negative Top Features': negative_top_features,

        'Top Deleted Score': top_deleted_score,
        'Retained Score': retained_score,

        'Comprehensiveness': comprehensiveness,
        'Sufficiency': sufficiency,

        'Absolute Comprehensiveness': abs(
            comprehensiveness
        ),

        'Absolute Sufficiency Error': abs(
            sufficiency
        ),

        'Mean Random Deleted Score': float(
            random_scores.mean()
        ),

        'Mean Random Score Drop': mean_random_drop,

        'Random Score Drop SD': float(
            random_drops.std(ddof=1)
        ),

        'Mean Random Absolute Change': (
            mean_random_absolute_change
        ),

        'Signed Top Deletion Advantage': (
            signed_advantage
        ),

        'Absolute Top Deletion Advantage': (
            absolute_advantage
        ),

        'Top Absolute Change Exceeds Random Mean': (
            bool(absolute_advantage > 0)
        ),

        'Top Deleted Prediction': top_deleted_pred,

        'Top Deletion Prediction Flip': (
            original_pred != top_deleted_pred
        ),

        'Retained Prediction': retained_pred,

        'Retained Prediction Matches Original': (
            original_pred == retained_pred
        ),

        'Random Deletion Prediction Flip Rate (%)': (
            100 * float(
                (random_pred != original_pred).mean()
            )
        ),

        'Random Draws': RANDOM_DRAWS,

        'Unique Random Subsets': unique_random_subsets,

        'Original Token Count': int(
            row['Original Token Count']
        ),

        'Truncated': bool(row['Truncated'])
    })

    # Store private perturbation-level evidence

    experiment_records = [

        ('Original', 0, [], original_score),

        (
            'Top deletion',
            0,
            top_ids.tolist(),
            top_deleted_score
        ),

        (
            'Top retention',
            0,
            remaining_ids.tolist(),
            retained_score
        )
    ]

    experiment_records.extend([

        (
            'Random deletion',
            draw + 1,
            random_sets[draw].tolist(),
            float(random_scores[draw])
        )

        for draw in range(RANDOM_DRAWS)
    ])

    for operation, draw, removed_ids, score in (
        experiment_records
    ):

        perturbation_rows.append({

            'Case ID': case_id,
            'Target Label': target,
            'Operation': operation,
            'Draw': draw,

            'Removed Feature IDs': json.dumps(
                removed_ids
            ),

            'Removed Feature Count': len(
                removed_ids
            ),

            'Target Score': score,

            'Score Drop': (
                original_score - score
            ),

            'Predicted Target': int(
                score >= cutoff
            )
        })

    print(
        case_id,
        '| Target:', target,
        '| Top drop:', round(comprehensiveness, 4),
        '| Random drop:', round(mean_random_drop, 4),
        '| Sufficiency:', round(sufficiency, 4),
        '| Absolute advantage:', round(
            absolute_advantage, 4
        ),
        flush=True
    )

torch.cuda.synchronize()

experiment_seconds = (
    time.perf_counter() - experiment_start
)

# Part B: Verify all completed experiments

faithfulness_per_case = pd.DataFrame(
    faithfulness_rows
)

perturbation_details = pd.DataFrame(
    perturbation_rows
)

assert len(faithfulness_per_case) == 20
assert faithfulness_per_case['Case ID'].is_unique

assert len(perturbation_details) == 460

assert perturbation_details.groupby(
    'Case ID'
).size().eq(23).all()

assert len(
    perturbation_details[
        perturbation_details['Operation'] ==
        'Random deletion'
    ]
) == 400

assert faithfulness_per_case[
    'Original Score Error'
].le(0.002).all()

np.testing.assert_allclose(

    faithfulness_per_case['Comprehensiveness'],

    faithfulness_per_case['Original Score'] -
    faithfulness_per_case['Top Deleted Score'],

    atol=1e-12
)

np.testing.assert_allclose(

    faithfulness_per_case['Sufficiency'],

    faithfulness_per_case['Original Score'] -
    faithfulness_per_case['Retained Score'],

    atol=1e-12
)

print('\nAll faithfulness experiments: Verified')


# Part C: Summarize results

def summarize_faithfulness(frame, level, group):

    return {

        'Level': level,
        'Group': group,
        'Cases': len(frame),

        'Mean Comprehensiveness': frame[
            'Comprehensiveness'
        ].mean(),

        'Mean Random Score Drop': frame[
            'Mean Random Score Drop'
        ].mean(),

        'Mean Signed Advantage': frame[
            'Signed Top Deletion Advantage'
        ].mean(),

        'Mean Absolute Top Change': frame[
            'Absolute Comprehensiveness'
        ].mean(),

        'Mean Absolute Random Change': frame[
            'Mean Random Absolute Change'
        ].mean(),

        'Mean Absolute Advantage': frame[
            'Absolute Top Deletion Advantage'
        ].mean(),

        'Mean Sufficiency': frame[
            'Sufficiency'
        ].mean(),

        'Mean Absolute Sufficiency Error': frame[
            'Absolute Sufficiency Error'
        ].mean(),

        'Top Deletion Outperforms Random Cases': int(
            frame[
                'Top Absolute Change Exceeds Random Mean'
            ].sum()
        ),

        'Top Deletion Prediction Flips': int(
            frame[
                'Top Deletion Prediction Flip'
            ].sum()
        ),

        'Retained Prediction Matches': int(
            frame[
                'Retained Prediction Matches Original'
            ].sum()
        )
    }


summary_rows = [

    summarize_faithfulness(
        faithfulness_per_case,
        'Overall',
        'All cases'
    )
]

for group, frame in faithfulness_per_case.groupby(
    'Target Outcome'
):

    summary_rows.append(
        summarize_faithfulness(
            frame,
            'Outcome',
            group
        )
    )

for group, frame in faithfulness_per_case.groupby(
    'Target Label'
):

    summary_rows.append(
        summarize_faithfulness(
            frame,
            'Label',
            group
        )
    )

faithfulness_summary = pd.DataFrame(
    summary_rows
)

assert len(faithfulness_summary) == 11


# Part D: Save result tables

per_case_path = (
    TABLES_DIR / 'T03_faithfulness_per_case.csv'
)

summary_path = (
    TABLES_DIR / 'faithfulness_summary.csv'
)

perturbation_path = (
    PRIVATE_DIR / 'faithfulness_perturbation_details.csv'
)

save_csv_atomic(
    per_case_path,
    faithfulness_per_case
)

save_csv_atomic(
    summary_path,
    faithfulness_summary
)

save_csv_atomic(
    perturbation_path,
    perturbation_details
)

assert len(pd.read_csv(per_case_path)) == 20
assert len(pd.read_csv(summary_path)) == 11
assert len(pd.read_csv(perturbation_path)) == 460


# Part E: Verify frozen model and protocol

assert calculate_sha256(
    FROZEN_MANIFEST_PATH
) == EXPECTED_FROZEN_HASH

assert calculate_sha256(
    REGISTRY_PATH
) == frozen_manifest['registry']['sha256']

assert calculate_sha256(
    DECISION_LOG_PATH
) == frozen_manifest['decision_log_sha256']

assert calculate_sha256(
    selected_cases_path
) == SELECTION_HASH

assert calculate_sha256(
    attribution_path
) == lime_completion['attribution_sha256']

assert model_state_hash(model) == (
    primary_model_entry['model_state_sha256']
)

assert not model.training


# Record final faithfulness inference completion

faithfulness_inventory = []

for path in [

    per_case_path,
    summary_path,
    perturbation_path,
    faithfulness_config_path
]:

    faithfulness_inventory.append({

        'path': str(
            path.relative_to(PROJECT_ROOT)
        ),

        'size_bytes': int(
            path.stat().st_size
        ),

        'sha256': calculate_sha256(path)
    })

for record in faithfulness_inventory:
    verify_file(record)

overall = faithfulness_summary[
    faithfulness_summary['Level'] == 'Overall'
].iloc[0]

faithfulness_completion = {

    'status': 'Completed',

    'primary_run_id': PRIMARY_RUN_ID,

    'evaluated_cases': 20,
    'target_labels': 6,

    'top_k': TOP_K,
    'random_draws_per_case': RANDOM_DRAWS,

    'random_deletion_evaluations': 400,
    'top_deletion_evaluations': 20,
    'top_retention_evaluations': 20,
    'original_replay_evaluations': 20,

    'total_evaluation_records': 460,

    'mean_comprehensiveness': float(
        overall['Mean Comprehensiveness']
    ),

    'mean_random_score_drop': float(
        overall['Mean Random Score Drop']
    ),

    'mean_absolute_top_change': float(
        overall['Mean Absolute Top Change']
    ),

    'mean_absolute_random_change': float(
        overall['Mean Absolute Random Change']
    ),

    'mean_absolute_advantage': float(
        overall['Mean Absolute Advantage']
    ),

    'mean_sufficiency': float(
        overall['Mean Sufficiency']
    ),

    'mean_absolute_sufficiency_error': float(
        overall['Mean Absolute Sufficiency Error']
    ),

    'top_deletion_outperforms_random_cases': int(
        overall['Top Deletion Outperforms Random Cases']
    ),

    'maximum_original_score_error': float(
        faithfulness_per_case[
            'Original Score Error'
        ].max()
    ),

    'inference_seconds': experiment_seconds,

    'frozen_manifest_sha256': EXPECTED_FROZEN_HASH,
    'selection_sha256': SELECTION_HASH,

    'lime_generation_completion_sha256': (
        calculate_sha256(lime_completion_path)
    ),

    'artifact_inventory': faithfulness_inventory,

    'model_training_performed': False,
    'thresholds_changed': False,
    'test_driven_selection_performed': False,

    'additional_inference_performed': True,
    'frozen_model_state_verified': True,

    'interpretations_completed': False,

    'completed_at': datetime.now(
        timezone.utc
    ).isoformat()
}

save_json_atomic(

    LOGS_DIR / 'faithfulness_completion_summary.json',

    faithfulness_completion
)


# Part F: Display research results

print('\nFaithfulness results per case')

display(

    faithfulness_per_case[[
        'Case ID',
        'Target Label',
        'Target Outcome',
        'Original Score',
        'Top Deleted Score',
        'Comprehensiveness',
        'Mean Random Score Drop',
        'Absolute Top Deletion Advantage',
        'Retained Score',
        'Sufficiency',
        'Top Deletion Prediction Flip'
    ]].round(6)
)

print('\nOverall faithfulness summary')

display(
    faithfulness_summary[
        faithfulness_summary['Level'] == 'Overall'
    ].round(6)
)

print('\nFaithfulness by prediction outcome')

display(
    faithfulness_summary[
        faithfulness_summary['Level'] == 'Outcome'
    ].round(6)
)

print('\nFaithfulness by target label')

display(
    faithfulness_summary[
        faithfulness_summary['Level'] == 'Label'
    ].round(6)
)

print('\nCell 5 summary')
print('Evaluated cases:', len(faithfulness_per_case))
print('Top deletion evaluations: 20')
print('Random deletion evaluations: 400')
print('Top retention evaluations: 20')
print('Original replay evaluations: 20')
print('Total evaluation records:', len(perturbation_details))
print('Summary records:', len(faithfulness_summary))
print('Experiment seconds:', round(experiment_seconds, 3))
print('Verified artifact inventory:', len(faithfulness_inventory))
print('Model weights: Unchanged')
print('Model training: None')
print('Frozen protocol: Unchanged')
print('Cell 5 completed successfully.')

In [ ]:
# Cell 6A: Final figures and interpretation preparation

from PIL import Image
from datetime import datetime, timezone

# Restore completed experiment records

lime_completion_path = (
    LOGS_DIR / 'lime_generation_completion.json'
)

faithfulness_completion_path = (
    LOGS_DIR / 'faithfulness_completion_summary.json'
)

lime_completion = read_json(lime_completion_path)

faithfulness_completion = read_json(
    faithfulness_completion_path
)

assert lime_completion['status'] == 'Completed'
assert faithfulness_completion['status'] == 'Completed'

assert lime_completion['explained_cases'] == 20
assert faithfulness_completion['evaluated_cases'] == 20

assert faithfulness_completion[
    'total_evaluation_records'
] == 460

for record in [
    lime_completion,
    faithfulness_completion
]:

    assert record['frozen_manifest_sha256'] == (
        EXPECTED_FROZEN_HASH
    )

    assert not record['thresholds_changed']
    assert not record['model_training_performed']

assert calculate_sha256(
    lime_completion_path
) == faithfulness_completion[
    'lime_generation_completion_sha256'
]

# Verify previous experiment artifacts

for record in lime_completion['artifact_inventory']:
    verify_file(record)

for record in faithfulness_completion['artifact_inventory']:
    verify_file(record)

print('LIME artifacts verified: 83')
print('Faithfulness artifacts verified: 4')

# Restore selected comments and configuration

selection_config_path = (
    RUN_CONFIGS_DIR / 'explanation_case_selection.json'
)

selection_config = read_json(selection_config_path)

selected_cases_path = (
    PROJECT_ROOT /
    selection_config['selected_cases_path']
)

SELECTION_HASH = (
    'b440129b98122fbcb89f1fc039a88fb38'
    '27affe3ea3ff6a7be64670721d45f92'
)

assert calculate_sha256(
    selected_cases_path
) == SELECTION_HASH

assert selection_config[
    'selected_cases_sha256'
] == SELECTION_HASH

assert lime_completion['selection_sha256'] == SELECTION_HASH

assert faithfulness_completion[
    'selection_sha256'
] == SELECTION_HASH

selected_cases = pd.read_csv(
    selected_cases_path,
    keep_default_na=False,
    float_precision='round_trip'
)

attribution_path = (
    PROJECT_ROOT / lime_completion['attribution_path']
)

feature_attributions = pd.read_csv(
    attribution_path,
    keep_default_na=False,
    dtype={'Feature': str},
    float_precision='round_trip'
)

lime_case_summary = pd.read_csv(
    PROJECT_ROOT / lime_completion['summary_path'],
    float_precision='round_trip'
)

faithfulness_per_case = pd.read_csv(
    TABLES_DIR / 'T03_faithfulness_per_case.csv',
    float_precision='round_trip'
)

faithfulness_summary = pd.read_csv(
    TABLES_DIR / 'faithfulness_summary.csv'
)

assert len(selected_cases) == 20
assert len(lime_case_summary) == 20
assert len(faithfulness_per_case) == 20
assert len(feature_attributions) == 196
assert len(faithfulness_summary) == 11

expected_ids = set(selected_cases['Case ID'])

for frame in [
    lime_case_summary,
    faithfulness_per_case
]:

    assert frame['Case ID'].is_unique

    assert set(frame['Case ID']) == expected_ids

assert set(
    feature_attributions['Case ID']
) == expected_ids

# Verify all selected source comments

for row in selected_cases.to_dict('records'):

    i = int(row['Source Row'])
    j = LABELS.index(row['Target Label'])

    assert row['Comment ID'] == str(test_ids[i])

    assert row['Comment Text'] == (
        test_df.iloc[i]['comment_text']
    )

    assert row['Target Score'] == primary_scores[i, j]

    assert row['Actual Target'] == y_test[i, j]

    assert row['Predicted Target'] == primary_pred[i, j]

print('Selected comments and explanation records: Verified')

# Align case-level data

faithfulness_per_case = (
    faithfulness_per_case
    .sort_values('Case ID')
    .reset_index(drop=True)
)

lime_case_summary = (
    lime_case_summary
    .sort_values('Case ID')
    .reset_index(drop=True)
)

np.testing.assert_array_equal(
    faithfulness_per_case['Case ID'],
    lime_case_summary['Case ID']
)

np.testing.assert_array_equal(
    faithfulness_per_case['Target Label'],
    lime_case_summary['Target Label']
)

# Figure saving helper

def save_explainability_figure(fig, filename):

    path = FIGURES_DIR / filename

    fig.tight_layout()

    fig.savefig(
        path,
        dpi=300,
        bbox_inches='tight',
        facecolor='white'
    )

    with Image.open(path) as image:

        assert image.format == 'PNG'
        image.verify()

    plt.show()
    plt.close(fig)

    print('Figure saved:', filename)


# F01: Top-feature deletion versus random deletion

x = np.arange(len(faithfulness_per_case))
width = 0.37

fig, axes = plt.subplots(
    2, 1,
    figsize=(14, 9),
    sharex=True
)

# Signed score changes

axes[0].bar(
    x - width / 2,
    faithfulness_per_case['Comprehensiveness'],
    width,
    label='LIME top-five deletion',
    color='#326796'
)

axes[0].bar(
    x + width / 2,
    faithfulness_per_case['Mean Random Score Drop'],
    width,
    label='Mean random deletion',
    color='#D99A46'
)

axes[0].set_ylabel('Signed score drop')

axes[0].set_title(
    'Comprehensiveness versus random deletion'
)

axes[0].axhline(
    0,
    color='black',
    linewidth=0.8
)

# Absolute score changes

axes[1].bar(
    x - width / 2,
    faithfulness_per_case['Absolute Comprehensiveness'],
    width,
    label='Absolute top deletion change',
    color='#326796'
)

axes[1].bar(
    x + width / 2,
    faithfulness_per_case['Mean Random Absolute Change'],
    width,
    label='Mean absolute random change',
    color='#D99A46'
)

axes[1].set_ylabel('Absolute score change')

axes[1].set_title(
    'Absolute perturbation sensitivity'
)

axes[1].set_xticks(x)

axes[1].set_xticklabels(
    faithfulness_per_case['Case ID'],
    rotation=45
)

axes[1].set_xlabel('Explanation case')

for ax in axes:

    ax.grid(axis='y', alpha=0.2)
    ax.set_axisbelow(True)
    ax.legend()

fig.suptitle(
    'LIME faithfulness: top features versus matched random features',
    fontsize=14
)

save_explainability_figure(
    fig,
    'F01_top_vs_random_deletion.png'
)


# F02: Surrogate fidelity and feature retention

fig, axes = plt.subplots(
    2, 1,
    figsize=(14, 9),
    sharex=True
)

# Local surrogate fidelity

axes[0].bar(
    x,
    lime_case_summary['Local Surrogate R2'],
    color='#326796'
)

mean_r2 = lime_case_summary[
    'Local Surrogate R2'
].mean()

axes[0].axhline(
    mean_r2,
    color='#D99A46',
    linestyle='--',
    label=f'Mean R2 = {mean_r2:.4f}'
)

axes[0].set_ylabel('Local surrogate R2')

axes[0].set_ylim(
    min(0, lime_case_summary['Local Surrogate R2'].min()),
    1.05
)

axes[0].set_title(
    'Local LIME surrogate fidelity'
)

axes[0].legend()

# Original and retained-feature scores

original = faithfulness_per_case[
    'Original Score'
].to_numpy()

retained = faithfulness_per_case[
    'Retained Score'
].to_numpy()

cutoffs = faithfulness_per_case[
    'Frozen Threshold'
].to_numpy()

for i in range(len(x)):

    axes[1].plot(
        [i, i],
        [original[i], retained[i]],
        color='#AAB5C2',
        linewidth=2,
        zorder=1
    )

axes[1].scatter(
    x,
    original,
    label='Original score',
    color='#326796',
    s=45,
    zorder=3
)

axes[1].scatter(
    x,
    retained,
    label='Top-five retained score',
    color='#C65D49',
    s=45,
    zorder=3
)

axes[1].scatter(
    x,
    cutoffs,
    label='Frozen target threshold',
    color='black',
    marker='_',
    s=130,
    zorder=4
)

axes[1].set_xticks(x)

axes[1].set_xticklabels(
    faithfulness_per_case['Case ID'],
    rotation=45
)

axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel('Target-label score')
axes[1].set_xlabel('Explanation case')

axes[1].set_title(
    'Score preservation after top-feature retention'
)

axes[1].legend(
    loc='upper center',
    bbox_to_anchor=(0.5, -0.20),
    ncol=3
)

for ax in axes:

    ax.grid(axis='y', alpha=0.2)
    ax.set_axisbelow(True)

fig.suptitle(
    'LIME fidelity and sufficiency analysis',
    fontsize=14
)

save_explainability_figure(
    fig,
    'F02_faithfulness_summary.png'
)


# Prepare interpretation worksheet

lime_lookup = lime_case_summary.set_index('Case ID')

faith_lookup = faithfulness_per_case.set_index(
    'Case ID'
)

review_rows = []

for row in selected_cases.to_dict('records'):

    case_id = row['Case ID']

    lime_row = lime_lookup.loc[case_id]
    faith_row = faith_lookup.loc[case_id]

    features = feature_attributions[
        feature_attributions['Case ID'] == case_id
    ].sort_values(
        ['Absolute Weight', 'Feature ID'],
        ascending=[False, True]
    )

    assert features['Target Label'].eq(
        row['Target Label']
    ).all()

    assert lime_row['Target Label'] == row['Target Label']
    assert faith_row['Target Label'] == row['Target Label']

    feature_records = []

    for feature in features.to_dict('records'):

        feature_records.append({

            'word': feature['Feature'],

            'weight': float(
                feature['LIME Weight']
            ),

            'visibility': feature[
                'Original Window Visibility'
            ],

            'total_occurrences': int(
                feature['Total Occurrences']
            ),

            'visible_occurrences': int(
                feature['Visible Occurrences']
            )
        })

    review_rows.append({

        'Case ID': case_id,
        'Stratum': row['Stratum'],

        'Comment Text': row['Comment Text'],

        'Target Label': row['Target Label'],
        'Target Outcome': row['Target Outcome'],

        'Actual Target': row['Actual Target'],
        'Predicted Target': row['Predicted Target'],

        'Actual Positive Labels': (
            row['Actual Positive Labels']
        ),

        'Predicted Positive Labels': (
            row['Predicted Positive Labels']
        ),

        'Frozen Score': row['Target Score'],
        'Frozen Threshold': row['Frozen Threshold'],

        'Identity Categories': (
            row['Identity Categories']
        ),

        'Identity Terms': row['Identity Terms'],

        'Original Token Count': (
            row['Original Token Count']
        ),

        'Truncated': row['Truncated'],

        'LIME Features': json.dumps(
            feature_records,
            ensure_ascii=False
        ),

        'Local Surrogate R2': (
            lime_row['Local Surrogate R2']
        ),

        'Top Deleted Score': (
            faith_row['Top Deleted Score']
        ),

        'Comprehensiveness': (
            faith_row['Comprehensiveness']
        ),

        'Mean Random Score Drop': (
            faith_row['Mean Random Score Drop']
        ),

        'Mean Random Absolute Change': (
            faith_row['Mean Random Absolute Change']
        ),

        'Retained Score': (
            faith_row['Retained Score']
        ),

        'Sufficiency': faith_row['Sufficiency'],

        'Retained Prediction Matches Original': (
            faith_row[
                'Retained Prediction Matches Original'
            ]
        ),

        'Written Interpretation': '',
        'Explanation Limitations': '',
        'Independent Human Review': 'Pending'
    })

interpretation_worksheet = pd.DataFrame(
    review_rows
)

assert len(interpretation_worksheet) == 20

assert interpretation_worksheet[
    'Case ID'
].is_unique

worksheet_path = (
    PRIVATE_DIR /
    'explanation_interpretation_worksheet.csv'
)

# Preserve an existing worksheet rather than overwriting it

if worksheet_path.exists():

    existing = pd.read_csv(
        worksheet_path,
        keep_default_na=False,
        float_precision='round_trip'
    )

    pd.testing.assert_frame_equal(
        existing,
        interpretation_worksheet,
        check_dtype=False,
        check_exact=True
    )

else:

    save_csv_atomic(
        worksheet_path,
        interpretation_worksheet
    )

WORKSHEET_HASH = calculate_sha256(
    worksheet_path
)

# Record preparation artifacts

preparation_inventory = []

for path in [

    worksheet_path,

    FIGURES_DIR / 'F01_top_vs_random_deletion.png',

    FIGURES_DIR / 'F02_faithfulness_summary.png'
]:

    preparation_inventory.append({

        'path': str(
            path.relative_to(PROJECT_ROOT)
        ),

        'size_bytes': int(
            path.stat().st_size
        ),

        'sha256': calculate_sha256(path)
    })

for record in preparation_inventory:
    verify_file(record)

preparation_summary = {

    'status': 'Completed',

    'selected_cases': 20,
    'interpretation_worksheet_rows': 20,
    'aggregate_figures': 2,

    'worksheet_path': str(
        worksheet_path.relative_to(PROJECT_ROOT)
    ),

    'worksheet_sha256': WORKSHEET_HASH,

    'selection_sha256': SELECTION_HASH,

    'frozen_manifest_sha256': EXPECTED_FROZEN_HASH,

    'lime_generation_completion_sha256': (
        calculate_sha256(lime_completion_path)
    ),

    'faithfulness_completion_sha256': (
        calculate_sha256(faithfulness_completion_path)
    ),

    'written_interpretations_completed': False,

    'independent_human_validation_completed': False,

    'model_training_performed': False,
    'thresholds_changed': False,
    'additional_inference_performed': False,

    'private_review_worksheet': True,

    'artifact_inventory': preparation_inventory,

    'completed_at': datetime.now(
        timezone.utc
    ).isoformat()
}

save_json_atomic(

    LOGS_DIR /
    'interpretation_preparation_summary.json',

    preparation_summary
)

# Final frozen protocol checks

assert calculate_sha256(
    FROZEN_MANIFEST_PATH
) == EXPECTED_FROZEN_HASH

assert calculate_sha256(
    REGISTRY_PATH
) == frozen_manifest['registry']['sha256']

assert calculate_sha256(
    DECISION_LOG_PATH
) == frozen_manifest['decision_log_sha256']

print('\nCell 6A summary')
print('LIME explanations restored: 20')
print('Feature attributions restored: 196')
print('Faithfulness cases restored: 20')
print('Aggregate figures saved: 2')
print('Interpretation worksheet rows: 20')
print('Worksheet SHA-256:', WORKSHEET_HASH)
print('Worksheet location:', worksheet_path)
print('Written interpretations: Pending')
print('Additional training and inference: None')
print('Frozen protocol: Unchanged')
print('Cell 6A completed successfully.')

In [ ]:
from google.colab import files

files.download(str(worksheet_path))

In [ ]:
# Cell 6B: Verify and integrate written interpretations

from datetime import datetime, timezone

# Restore interpretation preparation record

preparation_path = (
    LOGS_DIR / 'interpretation_preparation_summary.json'
)

preparation_summary = read_json(preparation_path)

assert preparation_summary['status'] == 'Completed'

assert preparation_summary[
    'interpretation_worksheet_rows'
] == 20

assert preparation_summary[
    'frozen_manifest_sha256'
] == EXPECTED_FROZEN_HASH

# Original and completed worksheets

original_worksheet_path = (
    PRIVATE_DIR /
    'explanation_interpretation_worksheet.csv'
)

completed_worksheet_path = (
    PRIVATE_DIR /
    'explanation_interpretation_worksheet_AI_assisted.csv'
)

assert completed_worksheet_path.is_file(), (
    'Upload the completed AI-assisted worksheet first.'
)

EXPECTED_ORIGINAL_HASH = (
    '3eec3147db47eba383ee4926fbe946c93'
    'd5a5f29521df98cfcda69afdcc6085f'
)

EXPECTED_COMPLETED_HASH = (
    'b540e670bb5c4397ca4cdc4d8644357fa'
    '3feac357d879333c8067e92579b804e'
)

assert calculate_sha256(
    original_worksheet_path
) == EXPECTED_ORIGINAL_HASH

assert calculate_sha256(
    completed_worksheet_path
) == EXPECTED_COMPLETED_HASH

assert preparation_summary[
    'worksheet_sha256'
] == EXPECTED_ORIGINAL_HASH

print('Original worksheet fingerprint: Verified')
print('Completed worksheet fingerprint: Verified')

# Verify that no original evidence has been changed

original_worksheet = pd.read_csv(
    original_worksheet_path,
    dtype=str,
    keep_default_na=False
)

completed_worksheet = pd.read_csv(
    completed_worksheet_path,
    dtype=str,
    keep_default_na=False
)

assert original_worksheet.shape == (20, 27)

assert completed_worksheet.shape == (20, 27)

assert original_worksheet.columns.tolist() == (
    completed_worksheet.columns.tolist()
)

assert completed_worksheet['Case ID'].is_unique

assert completed_worksheet['Case ID'].tolist() == [
    f'E{i:02d}' for i in range(1, 21)
]

interpretation_columns = [
    'Written Interpretation',
    'Explanation Limitations'
]

# Only the two interpretation columns may differ

pd.testing.assert_frame_equal(

    original_worksheet.drop(
        columns=interpretation_columns
    ),

    completed_worksheet.drop(
        columns=interpretation_columns
    ),

    check_exact=True
)

for column in interpretation_columns:

    assert original_worksheet[
        column
    ].str.strip().eq('').all()

    assert completed_worksheet[
        column
    ].str.strip().ne('').all()

assert completed_worksheet[
    'Independent Human Review'
].eq('Pending').all()

print('All original evidence fields: Unchanged')
print('Written interpretations: 20/20')
print('Case-specific limitations: 20/20')
print('Independent human review: Pending')

# Verify previous explanation and faithfulness artifacts

lime_completion_path = (
    LOGS_DIR / 'lime_generation_completion.json'
)

faithfulness_completion_path = (
    LOGS_DIR / 'faithfulness_completion_summary.json'
)

lime_completion = read_json(
    lime_completion_path
)

faithfulness_completion = read_json(
    faithfulness_completion_path
)

assert lime_completion['status'] == 'Completed'

assert faithfulness_completion['status'] == 'Completed'

assert calculate_sha256(
    lime_completion_path
) == preparation_summary[
    'lime_generation_completion_sha256'
]

assert calculate_sha256(
    faithfulness_completion_path
) == preparation_summary[
    'faithfulness_completion_sha256'
]

for record in lime_completion['artifact_inventory']:
    verify_file(record)

for record in faithfulness_completion['artifact_inventory']:
    verify_file(record)

for record in preparation_summary['artifact_inventory']:
    verify_file(record)

print('Previous explainability artifacts: Verified')

# Restore final numerical results

lime_case_summary = pd.read_csv(
    PROJECT_ROOT / lime_completion['summary_path'],
    float_precision='round_trip'
)

faithfulness_per_case = pd.read_csv(
    TABLES_DIR / 'T03_faithfulness_per_case.csv',
    float_precision='round_trip'
)

assert len(lime_case_summary) == 20

assert len(faithfulness_per_case) == 20

lime_lookup = lime_case_summary.set_index(
    'Case ID'
)

faith_lookup = faithfulness_per_case.set_index(
    'Case ID'
)

# Verify interpretation evidence against saved results

for row in completed_worksheet.to_dict('records'):

    case_id = row['Case ID']

    lime_row = lime_lookup.loc[case_id]

    faith_row = faith_lookup.loc[case_id]

    assert row['Target Label'] == lime_row['Target Label']

    assert row['Target Outcome'] == lime_row['Target Outcome']

    assert row['Target Label'] == faith_row['Target Label']

    assert row['Target Outcome'] == faith_row['Target Outcome']

    np.testing.assert_allclose(

        [
            float(row['Frozen Score']),
            float(row['Frozen Threshold']),
            float(row['Local Surrogate R2'])
        ],

        [
            lime_row['Frozen Score'],
            lime_row['Frozen Threshold'],
            lime_row['Local Surrogate R2']
        ],

        rtol=0,
        atol=1e-12
    )

    np.testing.assert_allclose(

        [
            float(row['Top Deleted Score']),
            float(row['Comprehensiveness']),
            float(row['Mean Random Score Drop']),
            float(row['Retained Score']),
            float(row['Sufficiency'])
        ],

        [
            faith_row['Top Deleted Score'],
            faith_row['Comprehensiveness'],
            faith_row['Mean Random Score Drop'],
            faith_row['Retained Score'],
            faith_row['Sufficiency']
        ],

        rtol=0,
        atol=1e-12
    )

print('All 20 interpretation evidence records: Verified')

# Create the required interpretation table

interpretation_table = completed_worksheet[[
    'Case ID',
    'Stratum',
    'Target Label',
    'Target Outcome',
    'Written Interpretation',
    'Explanation Limitations',
    'Independent Human Review'
]].copy()

interpretation_table['Interpretation Source'] = (
    'AI-assisted case-specific interpretation'
)

assert len(interpretation_table) == 20

interpretation_path = (
    PRIVATE_DIR / 'T04_interpretations.csv'
)

save_csv_atomic(
    interpretation_path,
    interpretation_table
)

pd.testing.assert_frame_equal(

    pd.read_csv(
        interpretation_path,
        dtype=str,
        keep_default_na=False
    ),

    interpretation_table,

    check_exact=True
)

# Create publication-safe interpretation status table

interpretation_status = completed_worksheet[[
    'Case ID',
    'Stratum',
    'Target Label',
    'Target Outcome',
    'Independent Human Review'
]].copy()

interpretation_status['Interpretation Completed'] = True

interpretation_status['Limitations Documented'] = True

interpretation_status['Interpretation Method'] = (
    'AI-assisted'
)

status_path = (
    TABLES_DIR / 'interpretation_status.csv'
)

save_csv_atomic(
    status_path,
    interpretation_status
)

# Select six report examples without using LIME fidelity
# to exclude difficult or unsuccessful explanations

report_case_design = [

    ('E04', 'Correct threat prediction'),

    ('E07', 'Toxic false positive'),

    ('E12', 'Missed threat prediction'),

    ('E15', 'Identity-related false positive'),

    ('E16', 'Context-dependent true negative'),

    ('E20', 'Truncated multilabel false positive')
]

report_cases = pd.DataFrame(
    report_case_design,
    columns=[
        'Case ID',
        'Report Selection Purpose'
    ]
)

report_cases = report_cases.merge(

    lime_case_summary[[
        'Case ID',
        'Target Label',
        'Target Outcome',
        'Frozen Score',
        'Frozen Threshold',
        'Local Surrogate R2'
    ]],

    on='Case ID',
    validate='one_to_one'
)

report_cases = report_cases.merge(

    faithfulness_per_case[[
        'Case ID',
        'Comprehensiveness',
        'Mean Random Score Drop',
        'Sufficiency',
        'Retained Prediction Matches Original',
        'Truncated'
    ]],

    on='Case ID',
    validate='one_to_one'
)

assert len(report_cases) == 6

report_cases_path = (
    TABLES_DIR / 'report_explanation_case_selection.csv'
)

save_csv_atomic(
    report_cases_path,
    report_cases
)

# Register final interpretation artifacts

interpretation_inventory = []

for path in [

    completed_worksheet_path,
    interpretation_path,
    status_path,
    report_cases_path

]:

    interpretation_inventory.append({

        'path': str(
            path.relative_to(PROJECT_ROOT)
        ),

        'size_bytes': int(
            path.stat().st_size
        ),

        'sha256': calculate_sha256(path)
    })

for record in interpretation_inventory:
    verify_file(record)

# Record interpretation completion

interpretation_completion = {

    'status': 'Completed',

    'interpretation_method': (
        'AI-assisted review of actual comments, '
        'signed LIME attributions, model predictions '
        'and faithfulness measurements'
    ),

    'reviewed_cases': 20,

    'written_interpretations_completed': True,

    'case_specific_limitations_completed': True,

    'independent_human_validation_completed': False,

    'independent_human_review_pending_cases': 20,

    'report_examples_selected': 6,

    'report_example_ids': (
        report_cases['Case ID'].tolist()
    ),

    'original_worksheet_sha256': (
        EXPECTED_ORIGINAL_HASH
    ),

    'completed_worksheet_sha256': (
        EXPECTED_COMPLETED_HASH
    ),

    'original_evidence_fields_unchanged': True,

    'frozen_manifest_sha256': EXPECTED_FROZEN_HASH,

    'selection_sha256': lime_completion[
        'selection_sha256'
    ],

    'preparation_summary_sha256': (
        calculate_sha256(preparation_path)
    ),

    'lime_generation_completion_sha256': (
        calculate_sha256(lime_completion_path)
    ),

    'faithfulness_completion_sha256': (
        calculate_sha256(faithfulness_completion_path)
    ),

    'artifact_inventory': interpretation_inventory,

    'private_raw_comment_interpretations': True,

    'model_training_performed': False,

    'thresholds_changed': False,

    'additional_inference_performed': False,

    'completed_at': datetime.now(
        timezone.utc
    ).isoformat()
}

completion_path = (
    LOGS_DIR /
    'interpretation_completion_summary.json'
)

save_json_atomic(

    completion_path,
    interpretation_completion
)

# Final frozen protocol checks

assert calculate_sha256(
    FROZEN_MANIFEST_PATH
) == EXPECTED_FROZEN_HASH

assert calculate_sha256(
    REGISTRY_PATH
) == frozen_manifest['registry']['sha256']

assert calculate_sha256(
    DECISION_LOG_PATH
) == frozen_manifest['decision_log_sha256']

assert calculate_sha256(
    original_worksheet_path
) == EXPECTED_ORIGINAL_HASH

assert calculate_sha256(
    completed_worksheet_path
) == EXPECTED_COMPLETED_HASH

print('\nSelected report explanation cases')

display(report_cases.round(6))

print('\nCell 6B summary')
print('Written interpretations: 20/20')
print('Case-specific limitations: 20/20')
print('Original evidence fields: Unchanged')
print('Interpretation method: AI-assisted')
print('Independent human review: Pending')
print('Report examples selected: 6')
print('Verified interpretation artifacts: 4')
print('Additional training and inference: None')
print('Frozen protocol: Unchanged')
print('Cell 6B completed successfully.')

In [ ]:
# Cell 7: Final explainability verification

from datetime import datetime, timezone
from PIL import Image

# Restore all completion records

selection_config_path = (
    RUN_CONFIGS_DIR /
    'explanation_case_selection.json'
)

lime_settings_path = (
    RUN_CONFIGS_DIR / 'lime_settings.json'
)

lime_completion_path = (
    LOGS_DIR / 'lime_generation_completion.json'
)

faithfulness_completion_path = (
    LOGS_DIR / 'faithfulness_completion_summary.json'
)

preparation_path = (
    LOGS_DIR / 'interpretation_preparation_summary.json'
)

interpretation_completion_path = (
    LOGS_DIR / 'interpretation_completion_summary.json'
)

selection_config = read_json(selection_config_path)
lime_settings = read_json(lime_settings_path)
lime_completion = read_json(lime_completion_path)

faithfulness_completion = read_json(
    faithfulness_completion_path
)

preparation_summary = read_json(preparation_path)

interpretation_completion = read_json(
    interpretation_completion_path
)

completion_records = [
    selection_config,
    lime_completion,
    faithfulness_completion,
    preparation_summary,
    interpretation_completion
]

for record in completion_records:

    assert record['status'] == 'Completed'

    assert record['frozen_manifest_sha256'] == (
        EXPECTED_FROZEN_HASH
    )

    assert not record['model_training_performed']
    assert not record['thresholds_changed']

print('All explainability stage completion records: Verified')

# Verify frozen selection

SELECTION_HASH = (
    'b440129b98122fbcb89f1fc039a88fb38'
    '27affe3ea3ff6a7be64670721d45f92'
)

selected_cases_path = (
    PROJECT_ROOT /
    selection_config['selected_cases_path']
)

assert calculate_sha256(
    selected_cases_path
) == SELECTION_HASH

assert selection_config[
    'selected_cases_sha256'
] == SELECTION_HASH

for record in [
    lime_completion,
    faithfulness_completion,
    preparation_summary,
    interpretation_completion
]:

    assert record['selection_sha256'] == SELECTION_HASH

assert calculate_sha256(
    lime_settings_path
) == lime_completion['lime_settings_sha256']

# Verify the chain of completion records

assert calculate_sha256(
    lime_completion_path
) == faithfulness_completion[
    'lime_generation_completion_sha256'
]

assert calculate_sha256(
    lime_completion_path
) == preparation_summary[
    'lime_generation_completion_sha256'
]

assert calculate_sha256(
    faithfulness_completion_path
) == preparation_summary[
    'faithfulness_completion_sha256'
]

assert calculate_sha256(
    preparation_path
) == interpretation_completion[
    'preparation_summary_sha256'
]

assert calculate_sha256(
    lime_completion_path
) == interpretation_completion[
    'lime_generation_completion_sha256'
]

assert calculate_sha256(
    faithfulness_completion_path
) == interpretation_completion[
    'faithfulness_completion_sha256'
]

print('Frozen selection and completion chain: Verified')

# Consolidate artifact inventory

inventory_map = {}

def register_record(record):

    path = verify_file(record)

    key = str(path.relative_to(PROJECT_ROOT))

    if key in inventory_map:
        assert inventory_map[key] == record

    else:
        inventory_map[key] = record


def register_path(path):

    path = Path(path)

    assert path.is_file()
    assert path.stat().st_size > 0

    record = {
        'path': str(path.relative_to(PROJECT_ROOT)),
        'size_bytes': int(path.stat().st_size),
        'sha256': calculate_sha256(path)
    }

    register_record(record)


for completion in [
    lime_completion,
    faithfulness_completion,
    preparation_summary,
    interpretation_completion
]:

    for record in completion['artifact_inventory']:
        register_record(record)

assert len(inventory_map) == 94

print('Previously inventoried artifacts: 94 verified')

# Verify selection metadata

metadata_path = (
    PROJECT_ROOT /
    selection_config['public_metadata_path']
)

assert calculate_sha256(
    metadata_path
) == selection_config['public_metadata_sha256']

selection_summary_path = (
    TABLES_DIR / 'explanation_selection_summary.csv'
)

assert calculate_sha256(
    selection_summary_path
) == selection_config['selection_summary_sha256']

# Restore final result tables

selected_cases = pd.read_csv(
    selected_cases_path,
    keep_default_na=False,
    float_precision='round_trip'
)

feature_attributions = pd.read_csv(
    PROJECT_ROOT / lime_completion['attribution_path'],
    keep_default_na=False,
    dtype={'Feature': str},
    float_precision='round_trip'
)

lime_case_summary = pd.read_csv(
    PROJECT_ROOT / lime_completion['summary_path'],
    float_precision='round_trip'
)

fidelity_summary = pd.read_csv(
    TABLES_DIR / 'lime_fidelity_summary.csv'
)

faithfulness_per_case = pd.read_csv(
    TABLES_DIR / 'T03_faithfulness_per_case.csv',
    float_precision='round_trip'
)

faithfulness_summary = pd.read_csv(
    TABLES_DIR / 'faithfulness_summary.csv'
)

perturbation_details = pd.read_csv(
    PRIVATE_DIR / 'faithfulness_perturbation_details.csv'
)

interpretation_table = pd.read_csv(
    PRIVATE_DIR / 'T04_interpretations.csv',
    keep_default_na=False
)

report_cases = pd.read_csv(
    TABLES_DIR / 'report_explanation_case_selection.csv'
)

assert len(selected_cases) == 20
assert len(feature_attributions) == 196
assert len(lime_case_summary) == 20
assert len(fidelity_summary) == 1
assert len(faithfulness_per_case) == 20
assert len(faithfulness_summary) == 11
assert len(perturbation_details) == 460
assert len(interpretation_table) == 20
assert len(report_cases) == 6

expected_ids = {
    f'E{i:02d}' for i in range(1, 21)
}

for frame in [
    selected_cases,
    lime_case_summary,
    faithfulness_per_case,
    interpretation_table
]:

    assert frame['Case ID'].is_unique
    assert set(frame['Case ID']) == expected_ids

assert set(
    feature_attributions['Case ID']
) == expected_ids

assert selected_cases['Comment ID'].is_unique

assert selected_cases[
    'Normalized Group ID'
].is_unique

assert set(
    selected_cases['Target Label']
) == set(LABELS)

print('Case coverage and result table integrity: Verified')

# Verify original selected evidence

for row in selected_cases.to_dict('records'):

    i = int(row['Source Row'])
    j = LABELS.index(row['Target Label'])

    assert row['Comment ID'] == str(test_ids[i])

    assert row['Comment Text'] == (
        test_df.iloc[i]['comment_text']
    )

    assert row['Target Score'] == primary_scores[i, j]

    assert row['Frozen Threshold'] == thresholds[j]

    assert row['Actual Target'] == y_test[i, j]

    assert row['Predicted Target'] == primary_pred[i, j]

print('All 20 original prediction records: Verified')

# Verify LIME attribution counts

assert lime_case_summary[
    'Perturbation Samples'
].eq(1000).all()

assert lime_case_summary[
    'Perturbation Samples'
].sum() == 20000

assert lime_case_summary[
    'Original Score Error'
].le(0.002).all()

assert np.isfinite(
    lime_case_summary['Local Surrogate R2']
).all()

assert np.isfinite(
    feature_attributions['LIME Weight']
).all()

assert not feature_attributions.duplicated(
    ['Case ID', 'Feature ID']
).any()

feature_counts = (
    feature_attributions.groupby('Case ID')
    .size()
    .sort_index()
)

expected_feature_counts = (
    lime_case_summary.set_index('Case ID')
    ['Explained Features']
    .sort_index()
)

np.testing.assert_array_equal(
    feature_counts.to_numpy(),
    expected_feature_counts.to_numpy()
)

assert lime_case_summary[
    'Outside Window Features'
].sum() == 0

# Verify faithfulness experiment counts

expected_operations = {
    'Original': 20,
    'Top deletion': 20,
    'Top retention': 20,
    'Random deletion': 400
}

assert perturbation_details[
    'Operation'
].value_counts().to_dict() == expected_operations

assert perturbation_details.groupby(
    'Case ID'
).size().eq(23).all()

assert faithfulness_per_case['Top K'].eq(5).all()
assert faithfulness_per_case['Random Draws'].eq(20).all()

assert faithfulness_per_case[
    'Original Score Error'
].le(0.002).all()

np.testing.assert_allclose(
    faithfulness_per_case['Comprehensiveness'],

    faithfulness_per_case['Original Score'] -
    faithfulness_per_case['Top Deleted Score'],

    atol=1e-12,
    rtol=0
)

np.testing.assert_allclose(
    faithfulness_per_case['Sufficiency'],

    faithfulness_per_case['Original Score'] -
    faithfulness_per_case['Retained Score'],

    atol=1e-12,
    rtol=0
)

print('LIME and faithfulness numerical checks: Passed')

# Verify written interpretations

for column in [
    'Written Interpretation',
    'Explanation Limitations'
]:

    assert interpretation_table[
        column
    ].str.strip().ne('').all()

assert interpretation_table[
    'Independent Human Review'
].eq('Pending').all()

assert interpretation_completion[
    'written_interpretations_completed'
]

assert not interpretation_completion[
    'independent_human_validation_completed'
]

assert report_cases['Case ID'].is_unique

assert set(report_cases['Case ID']).issubset(
    expected_ids
)

print('Written interpretations: 20/20 verified')
print('Independent human validation: Pending')

# Verify all individual explanations

PRIVATE_FIGURES_DIR = PRIVATE_DIR / 'figures'
CASE_LOGS_DIR = PRIVATE_DIR / 'case_logs'

for case_id in sorted(expected_ids):

    figure_path = (
        PRIVATE_FIGURES_DIR /
        f'{case_id}_explanation.png'
    )

    html_path = (
        HTML_DIR / f'{case_id}_explanation.html'
    )

    case_log_path = (
        CASE_LOGS_DIR / f'{case_id}_summary.json'
    )

    case_summary = read_json(case_log_path)

    assert case_summary['status'] == 'Completed'
    assert case_summary['case_id'] == case_id

    assert calculate_sha256(
        figure_path
    ) == case_summary['figure_sha256']

    assert calculate_sha256(
        html_path
    ) == case_summary['html_sha256']

    with Image.open(figure_path) as image:

        assert image.format == 'PNG'
        assert image.width > 1000
        assert image.height > 1000

        image.verify()

    assert html_path.stat().st_size > 0

print('Individual PNG explanations: 20 verified')
print('HTML explanations: 20 verified')

# Verify aggregate figures

aggregate_figures = [
    'F01_top_vs_random_deletion.png',
    'F02_faithfulness_summary.png'
]

for filename in aggregate_figures:

    with Image.open(
        FIGURES_DIR / filename
    ) as image:

        assert image.format == 'PNG'
        assert image.width > 1000
        assert image.height > 1000

        image.verify()

print('Aggregate figures: 2 verified')

# Calculate final explainability metrics

mean_r2 = float(
    lime_case_summary['Local Surrogate R2'].mean()
)

median_r2 = float(
    lime_case_summary['Local Surrogate R2'].median()
)

minimum_r2 = float(
    lime_case_summary['Local Surrogate R2'].min()
)

maximum_r2 = float(
    lime_case_summary['Local Surrogate R2'].max()
)

mean_comprehensiveness = float(
    faithfulness_per_case['Comprehensiveness'].mean()
)

mean_random_drop = float(
    faithfulness_per_case['Mean Random Score Drop'].mean()
)

mean_absolute_top_change = float(
    faithfulness_per_case['Absolute Comprehensiveness'].mean()
)

mean_absolute_random_change = float(
    faithfulness_per_case[
        'Mean Random Absolute Change'
    ].mean()
)

mean_absolute_advantage = float(
    faithfulness_per_case[
        'Absolute Top Deletion Advantage'
    ].mean()
)

mean_sufficiency = float(
    faithfulness_per_case['Sufficiency'].mean()
)

mean_absolute_sufficiency = float(
    faithfulness_per_case[
        'Absolute Sufficiency Error'
    ].mean()
)

top_outperforms_random = int(
    faithfulness_per_case[
        'Top Absolute Change Exceeds Random Mean'
    ].sum()
)

retained_matches = int(
    faithfulness_per_case[
        'Retained Prediction Matches Original'
    ].sum()
)

prediction_flips = int(
    faithfulness_per_case[
        'Top Deletion Prediction Flip'
    ].sum()
)

absolute_change_ratio = (
    mean_absolute_top_change /
    mean_absolute_random_change
)

# Save concise final research findings

key_findings = pd.DataFrame([{

    'Model': 'RoBERTa',
    'Run ID': PRIMARY_RUN_ID,
    'Explainability Method': 'LIME',

    'Explained Cases': 20,
    'Target Labels Covered': 6,
    'Total Feature Attributions': 196,
    'LIME Samples': 20000,

    'Mean Local Surrogate R2': mean_r2,
    'Median Local Surrogate R2': median_r2,
    'Minimum Local Surrogate R2': minimum_r2,
    'Maximum Local Surrogate R2': maximum_r2,

    'Mean Comprehensiveness': mean_comprehensiveness,
    'Mean Random Score Drop': mean_random_drop,

    'Mean Absolute Top Change': mean_absolute_top_change,

    'Mean Absolute Random Change': (
        mean_absolute_random_change
    ),

    'Mean Absolute Advantage': mean_absolute_advantage,

    'Top To Random Absolute Change Ratio': (
        absolute_change_ratio
    ),

    'Mean Sufficiency': mean_sufficiency,

    'Mean Absolute Sufficiency Error': (
        mean_absolute_sufficiency
    ),

    'Top Deletion Outperforms Random Cases': (
        top_outperforms_random
    ),

    'Top Deletion Prediction Flips': prediction_flips,

    'Retained Prediction Matches': retained_matches,

    'Retained Prediction Match Rate (%)': (
        100 * retained_matches / 20
    ),

    'Written Interpretations': 20,
    'Independent Human Reviewed Cases': 0

}])

findings_path = (
    TABLES_DIR / 'explainability_key_findings.csv'
)

save_csv_atomic(
    findings_path,
    key_findings
)

# Add remaining configuration and summary artifacts

for path in [

    RUN_CONFIGS_DIR / 'explainability_setup.json',
    selection_config_path,
    lime_settings_path,

    selected_cases_path,
    metadata_path,
    selection_summary_path,

    TABLES_DIR / 'explanation_candidate_summary.csv',

    LOGS_DIR / 'lime_smoke_test_summary.json',

    lime_completion_path,
    faithfulness_completion_path,
    preparation_path,
    interpretation_completion_path,

    findings_path
]:

    register_path(path)

artifact_inventory = list(
    inventory_map.values()
)

for record in artifact_inventory:
    verify_file(record)

assert len({
    record['path']
    for record in artifact_inventory
}) == len(artifact_inventory)

print('Final artifact inventory: Verified')

# Verify frozen research protocol

assert calculate_sha256(
    FROZEN_MANIFEST_PATH
) == EXPECTED_FROZEN_HASH

assert calculate_sha256(
    REGISTRY_PATH
) == frozen_manifest['registry']['sha256']

assert calculate_sha256(
    DECISION_LOG_PATH
) == frozen_manifest['decision_log_sha256']

assert calculate_sha256(
    TEST_PATH
) == EXPECTED_TEST_HASH

assert calculate_sha256(
    primary_prediction_path
) == prediction_record['SHA256']

assert calculate_sha256(
    selected_cases_path
) == SELECTION_HASH

for record in primary_model_entry['model_files']:
    verify_file(record)

# Final Notebook 10 completion manifest

explainability_completion = {

    'notebook': '10_explainability',

    'status': 'Completed',

    'primary_run_id': PRIMARY_RUN_ID,

    'method': 'LIME',

    'explained_cases': 20,
    'unique_comments': 20,
    'unique_normalized_groups': 20,

    'target_labels_covered': 6,

    'target_outcome_counts': {
        key: int(value)
        for key, value in selected_cases[
            'Target Outcome'
        ].value_counts().items()
    },

    'truncated_cases': int(
        selected_cases['Truncated'].sum()
    ),

    'perturbations_per_case': 1000,
    'total_lime_samples': 20000,

    'feature_attribution_records': 196,

    'mean_local_surrogate_r2': mean_r2,
    'median_local_surrogate_r2': median_r2,
    'minimum_local_surrogate_r2': minimum_r2,
    'maximum_local_surrogate_r2': maximum_r2,

    'faithfulness_evaluation_records': 460,
    'random_deletion_evaluations': 400,

    'mean_comprehensiveness': mean_comprehensiveness,
    'mean_random_score_drop': mean_random_drop,

    'mean_absolute_top_change': mean_absolute_top_change,

    'mean_absolute_random_change': (
        mean_absolute_random_change
    ),

    'mean_absolute_advantage': mean_absolute_advantage,

    'top_to_random_absolute_change_ratio': (
        absolute_change_ratio
    ),

    'mean_sufficiency': mean_sufficiency,

    'mean_absolute_sufficiency_error': (
        mean_absolute_sufficiency
    ),

    'top_deletion_outperforms_random_cases': (
        top_outperforms_random
    ),

    'top_deletion_prediction_flips': prediction_flips,

    'retained_prediction_matches': retained_matches,

    'retained_prediction_match_rate': (
        retained_matches / 20
    ),

    'individual_explanation_figures': 20,
    'aggregate_figures': 2,
    'html_explanations': 20,

    'written_interpretations': 20,

    'interpretation_method': 'AI-assisted',

    'written_interpretations_completed': True,

    'independent_human_validation_completed': False,

    'independent_human_review_pending_cases': 20,

    'report_example_ids': (
        report_cases['Case ID'].tolist()
    ),

    'verified_artifacts': len(artifact_inventory),

    'artifact_inventory': artifact_inventory,

    'frozen_manifest_sha256': EXPECTED_FROZEN_HASH,
    'test_dataset_sha256': EXPECTED_TEST_HASH,
    'selection_sha256': SELECTION_HASH,

    'test_prediction_sha256': (
        prediction_record['SHA256']
    ),

    'lime_generation_completion_sha256': (
        calculate_sha256(lime_completion_path)
    ),

    'faithfulness_completion_sha256': (
        calculate_sha256(faithfulness_completion_path)
    ),

    'interpretation_completion_sha256': (
        calculate_sha256(interpretation_completion_path)
    ),

    'model_training_performed': False,
    'thresholds_changed': False,
    'test_driven_selection_performed': False,

    'additional_inference_performed': True,

    'additional_inference_scope': (
        'Original prediction replay, LIME perturbations '
        'and faithfulness experiments'
    ),

    'full_official_test_reinference_performed': False,

    'private_raw_comment_artifacts': True,

    'limitations': [

        'The 20 diagnostic cases are not representative '
        'of the entire test population.',

        'LIME weights describe a local surrogate '
        'rather than a causal model mechanism.',

        'Local surrogate fidelity varies across cases.',

        'Word removal can create unnatural text.',

        'High comprehensiveness does not guarantee '
        'high sufficiency.',

        'The random baseline uses 20 draws per case.',

        'Word deletion may change the truncation window.',

        'Scores are not assumed to be calibrated.',

        'Written interpretations are AI-assisted '
        'and require independent human review.'

    ],

    'next_notebook': '11_final_documentation',

    'completed_at': datetime.now(
        timezone.utc
    ).isoformat()
}

completion_path = (
    LOGS_DIR /
    'explainability_completion_summary.json'
)

save_json_atomic(
    completion_path,
    explainability_completion
)

# Display final completion summary

print('\nFinal explainability research findings')

display(key_findings.T)

print('\nNotebook 10 completion summary')

for key, value in explainability_completion.items():

    if key not in [
        'artifact_inventory',
        'limitations'
    ]:

        print(f'{key}: {value}')

print('\nAll explainability artifacts: Verified')
print('Frozen model weights and thresholds: Unchanged')
print('Additional training and inference in this cell: None')
print('Independent human review: Pending')
print('Cell 7 completed successfully.')
print('Notebook 10 completed successfully.')